# Figures — cleaned pipeline


## Setup

In [ ]:
from pathlib import Path

# Resolve the repo root from this notebook's location (repo_root/notebooks/this_file.ipynb)
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

import numpy as np
import pandas as pd
import re
import pyranges as pr
import glob
import os
import time
from pathlib import Path

import sys
sys.path.insert(1, str(REPO_ROOT / "src"))
import utils as mf
import pyBigWig as pbw
import logomaker as lm
import matplotlib.pyplot as plt
import matplotlib as mpl
from functools import wraps
from pathlib import Path
from scipy.signal import find_peaks

from scipy import sparse
from scipy.stats import hypergeom

from Bio import SeqIO
from statsmodels.stats.multitest import multipletests
from sklearn.metrics.pairwise import cosine_similarity

from joblib import Parallel, delayed

mpl_style = REPO_ROOT / "notebooks" / "genome_research.mplstyle"
mpl.rcParams.update(mpl.rcParamsDefault)
mpl.style.use(mpl_style)

plt.rcParams.update({
    # Font — picks the first one it finds installed
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],

    # Force every text element to the same size
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.title_fontsize": 10,
    "figure.titlesize": 10,

    # Keep text as real, editable text in the SVG — not outlines
    "svg.fonttype": "none",
})

jaspar_colors = {
    'A': '#FF0000',  # rouge
    'C': '#0000FF',  # bleu
    'G': '#FFA500',  # orange
    'T': '#008000'   # vert
}

def extract_motif_name_pairs(row):
    motif_ids = re.findall(r"(MA\d+\.\d+)", row["id"])
    tf_names = [name.strip() for name in row["name"].split(",")]
    if len(motif_ids) != len(tf_names):
        raise ValueError(f"Nombre de motifs ≠ nombre de noms dans la ligne cluster {row['cluster']}")
    return list(zip(motif_ids, tf_names))

def parse_meme_file(filepath):
    motifs = []
    with open(filepath, "r") as f:
        lines = f.readlines()

    motif = None
    pwm_lines = []

    for line in lines:
        if line.startswith("MOTIF"):
            if motif:
                motif["pwm"] = pwm_lines
                motifs.append(motif)

            parts = line.strip().split(maxsplit=2)
            raw = parts[2] if len(parts) > 2 else ""

            motif_id = None
            alt_name = None

            # 1) JASPAR : MA0139.2-CTCF
            m = re.match(r"^(MA\d+\.\d+)-(.*)$", raw)
            if m:
                motif_id = m.group(1)
                alt_name = m.group(2)

            else:
                # 2) De novo (STREME/MEME): with or without a leading rank
                # Ex:
                #   1-CCASHAGRGGGCRBY-STREME-1
                #   1-CCASHAGRGGG-MEME-2
                #   CCASHAGRGGG-MEME-1
                m = re.match(r"^(?:\d+-)?(.+?)-(STREME|MEME)-(\d+)$", raw)
                if m:
                    motif_id = m.group(1)              # seed/consensus
                    alt_name = f"{m.group(2)}-{m.group(3)}"  # STREME-1 / MEME-2
                else:
                    # fallback
                    motif_id = parts[1] if len(parts) > 1 else raw
                    alt_name = raw

            motif = {
                "motif_id": motif_id,
                "alt_name": alt_name,
                "w": None,
                "nsites": None,
                "evalue": None,
                "pwm": []
            }
            pwm_lines = []

        elif line.startswith("letter-probability matrix"):
            match = re.search(r"w=\s*(\d+)\s+nsites=\s*([\d.]+)\s+E=\s*([\deE\-.+]+)", line)
            if match and motif:
                motif["w"] = int(match.group(1))
                motif["nsites"] = float(match.group(2))
                motif["evalue"] = float(match.group(3))

        elif re.match(r"^\s*[\d.eE+-]", line) and motif:
            pwm_lines.append([float(x) for x in line.strip().split()])

    if motif:
        motif["pwm"] = pwm_lines
        motifs.append(motif)

    return motifs

def load_from_bw(path, chrom, start=0, end=-1):
    with pbw.open(str(path)) as bw:
        return bw.values(chrom, start, end, numpy=True)

def save_fig_decorator(name, path=None): 
    def decorator(plot_fun):
        @wraps(plot_fun)
        def wrapper(*args, **kwargs):
            fig, axs = plot_fun(*args, **kwargs)
            save_dir = path if path is not None else fig_path
            fig.savefig(save_dir / Path(f"{name}.svg"))
            return fig, axs
        return wrapper
    return decorator        
    


## Path configuration

Every external file the notebook reads lives outside this repo on the original machine. Below, everything is expressed relative to `REPO_ROOT` using the layout already implied by `README.md` (`data/` for raw reference data and assay bigwigs, `demo/test/<run_name>/` for a specific trained model's outputs). **You'll need to place the corresponding files at these locations** (or edit the dict) before running the rest of the notebook.


In [ ]:
DATA_DIR = REPO_ROOT / "data"
DEMO_DIR = REPO_ROOT / "demo"

PATHS = {
    # --- Reference genome / annotation data (place under data/) ---
    "chrom_sizes":        DATA_DIR / "sequences" / "mm10.chrom.sizes",
    "repeat_masker":      DATA_DIR / "sequences" / "repeat_masker",
    "jaspar_clusters_tsv": DATA_DIR / "jaspar_clusters2026.tsv",
    "jaspar_meme":        DATA_DIR / "sequences" / "JASPAR2026_vertebrates.meme",

    # --- RNA-seq / TSS annotations ---
    "rnaseq_minus_bw":    DATA_DIR / "rnaseq" / "ENCFF373ZMQ_minusStrand.bigWig",
    "rnaseq_plus_bw":     DATA_DIR / "rnaseq" / "ENCFF907JEV_plusStrand.bigWig",
    "tss_coord_bed":      DATA_DIR / "refTSS_3.3" / "refTSS_v3.3_mouse_coordinate.mm10.bed",
    "tss_classification": DATA_DIR / "refTSS_3.3" / "TSS.classification.mm10",
    "tss_annotation":     DATA_DIR / "refTSS_3.3" / "refTSS_v3.3_mouse_annotation.txt",
    "rna_tpm_report":     DATA_DIR / "rnaseq" / "rna_expression_report_2025_6_30_15h_3m.tsv",

    # --- CTCF ChIP-seq / JASPAR motif sites ---
    "ctcf_chip_bed":      DATA_DIR / "CTCF" / "all" / "ctcf.bed",
    "ctcf_jaspar_sites":  DATA_DIR / "JASPAR_TFBs_2024" / "MA0139.2.tsv.gz",

    # --- Chromatin accessibility / labels (place under data/) ---
    "dnase_bw":           DATA_DIR / "dnase" / "labels" / "DNase_ENCFF672DJH.bw",
    "atac1_bw":           DATA_DIR / "atac" / "GSM3719303_ATAC_1_RPKM.bw",
    "atac2_bw":           DATA_DIR / "atac" / "GSM3719304_ATAC_2_RPKM.bw",

    # --- Nucleosome labels and predictions (corrected: these live under demo/, not data/) ---
    "mnase_lab_bw":                 DEMO_DIR / "labels" / "sparse_A_16_gaussiansmoothed.bw",
    "chem_lab_bw":                  DEMO_DIR / "labels" / "GSM2183909_Chemical_NCPscore_mm10_SMOOTHED.bw",
    "mnase_pred_bw":                DEMO_DIR / "test" / "MNase_17chroms_CORRECTED_1.5prctCovered_globalWeightsOptim_20260303_142239" / "_ism_outputs" / "predictions.bw",
    "chem_pred_bw":                 DEMO_DIR / "test" / "Chemical_17chroms_CORRECTED_global_weights_20260304_205232" / "_ism_outputs" / "predictions.bw",
    "mnase_train_mutasome_bw":      DEMO_DIR / "test" / "MNase_17chroms_CORRECTED_1.5prctCovered_globalWeightsOptim_20260303_142239" / "_ism_outputs" / "ism.bw",
    "chem_train_mutasome_bw":       DEMO_DIR / "test" / "Chemical_17chroms_CORRECTED_global_weights_20260304_205232" / "_ism_outputs" / "ism.bw",
    "nprs_df":                      DEMO_DIR / "test" / "nprs.csv",

    # --- Trained-model run directories (place under demo/test/) ---
    "mnase_run_dir":  DEMO_DIR / "test" / "MNase_17chroms_CORRECTED_1.5prctCovered_globalWeightsOptim_20260303_142239",
    "chem_run_dir":   DEMO_DIR / "test" / "Chemical_17chroms_CORRECTED_global_weights_20260304_205232",
    "xstreme_dir":    DEMO_DIR / "test" / "xstreme250326_default_maxw30_JASPAR2026",
    "fimo_by_motif_single_csv": DEMO_DIR / "test" / "xstreme250326_default_maxw30_JASPAR2026" / "fimo_by_motif" / "_plots" / "combined_single.csv",
    "crossmut_bw":    DEMO_DIR / "test" / "crossmut.bw",
    "npr_bed_path":   DEMO_DIR / "test" / "nprs.csv",

    # FIXME: original paths were "demo/testmnase_pred_wavelets" / "demo/testchemical_pred_wavelets"
    # (looked like a missing path separator / typo) - verify the intended run directory.
    "wavelets_bw":        DATA_DIR / "wavelets_lab_A16.bw",
    "wavelet_pred_mnase_bw": DEMO_DIR / "test" / "MNase_17chroms_CORRECTED_1.5prctCovered_globalWeightsOptim_20260303_142239" / "mnase_pred_wavelets",
    "wavelet_pred_chem_bw":  DEMO_DIR / "test" / "Chemical_17chroms_CORRECTED_global_weights_20260304_205232" / "chemical_pred_wavelets",

    # --- Figure output directory ---
    "fig_path": REPO_ROOT / "notebooks" / "FIGTEST",

    # --- Appendix-section-only paths (exploratory analyses, see bottom of notebook) ---
    "external_utils_dir":     None,  # FIXME: original notebook imported a `preprocess.default` module from
                                      # "/home/maxime/data/utils", which is NOT part of this repo. That module
                                      # needs to be located/vendored before the Appendix "mappability" section can run.
    "mnase_multimapped_bw":   DATA_DIR / "mnase" / "labels" / "mA_16.bw",
    "mappability_bw":         DEMO_DIR / "artifacts" / "mm10.mappability.150.bw",
    "mut_wave_df_csv":        DEMO_DIR / "mut_wave_df.csv",
}

PATHS["fig_path"].mkdir(parents=True, exist_ok=True)


## Reference annotations (repeat masker, JASPAR clusters, bigwig shortcuts) — load once, used throughout

In [ ]:
chrom_sizes_df = pd.read_csv(PATHS["chrom_sizes"], sep="\t", header=None)
chrom_sizes_df = chrom_sizes_df[chrom_sizes_df[0].isin([f"chr{i}" for i in range(1, 20)])]
GENOME_LEN = chrom_sizes_df[1].sum()

rm = pd.read_csv(PATHS["repeat_masker"], sep="\t")
rm_pr = pr.PyRanges(rm.rename(
    columns={
        "genoName": "Chromosome",
        "genoStart": "Start",
        "genoEnd": "End"
    }))

mnase_mut_bw = PATHS["mnase_run_dir"] / "_ism_outputs" / "ism.bw"
mnase_pred_bw = PATHS["mnase_pred_bw"]
mnase_lab_bw = PATHS["mnase_lab_bw"]

chem_mut_bw = PATHS["chem_run_dir"] / "_ism_outputs" / "ism.bw"
chem_pred_bw = PATHS["chem_pred_bw"]
chem_lab_bw = PATHS["chem_lab_bw"]

crossmut_bw = PATHS["crossmut_bw"]
npr_bed_path = PATHS["npr_bed_path"]

wavelets_bw = PATHS["wavelets_bw"]
wavelet_pred_mnase_bw = PATHS["wavelet_pred_mnase_bw"]
wavelet_pred_chem_bw = PATHS["wavelet_pred_chem_bw"]

DNase_bw = PATHS["dnase_bw"]
atac1_bw = PATHS["atac1_bw"]
atac2_bw = PATHS["atac2_bw"]

fig_path = PATHS["fig_path"]

# JASPAR cluster -> (motif_id, tf_name) lookup, used by the Motif-discovery / Table 1 section below
jaspar_clusters = pd.read_csv(PATHS["jaspar_clusters_tsv"], sep="\t")
jaspar_clusters.cluster = jaspar_clusters.cluster.str.lstrip("cluster_")
jaspar_clusters["pairs"] = jaspar_clusters.apply(extract_motif_name_pairs, axis=1)

jaspar_per_motif = jaspar_clusters.explode("pairs")
jaspar_per_motif[["motif_id", "tf_name"]] = pd.DataFrame(jaspar_per_motif["pairs"].tolist(), index=jaspar_per_motif.index)
jaspar_per_motif = jaspar_per_motif[["motif_id", "tf_name", "cluster"]]


## Figure 1B / SUPP1 A-D — Model vs. experiment correlation scatterplots

In [ ]:
def corrplot(x, y, axis_label=["x", "y"], ax_scale="norm", log_ax=False):
    fig, axs = plt.subplots(figsize=(2, 2))

    corr = np.corrcoef(x, y)[0,1]

    h, xedges, yedges, im = axs.hist2d(
        x, y,
        cmap="YlOrRd",
        bins=50,
        norm=mpl.colors.LogNorm()
    )

    im.set_edgecolor("none")
    im.set_linewidth(0.0)
    im.set_antialiased(False)
    im.set_rasterized(True)

    fig.colorbar(im, ax=axs)
    # cbar.ax.yaxis.set_major_locator(MaxNLocator(nbins=3))

    axs.text(0.05, 0.9, f"Pearson r: {corr:.2f}", transform=axs.transAxes)
    if ax_scale == "norm":
        axs.set_xlim(0, 1)
        axs.set_ylim(0, 1)
        axs.set_xticks([0, 0.5, 1.0])
        axs.set_yticks([0, 0.5, 1.0])
    else:
        axs.set_xlim(ax_scale[0][0], ax_scale[0][1])
        axs.set_ylim(ax_scale[1][0], ax_scale[1][1])

    if log_ax:
        axs.set_xscale("symlog", linthresh=1e-2)
        axs.set_yscale("symlog", linthresh=1e-2)

    axs.set_xlabel(axis_label[0])
    axs.set_ylabel(axis_label[1])

    return fig, axs

save_fig_decorator("histplot_mnase_chr19")(corrplot)(
    load_from_bw(mnase_lab_bw, "chr19"),
    load_from_bw(mnase_pred_bw, "chr19"),
    axis_label=["MNase-sequencing", "Prediction"],
    ax_scale = "norm"
)

save_fig_decorator("histplot_chem_chr19")(corrplot)(
    load_from_bw(chem_lab_bw, "chr19"),
    load_from_bw(chem_pred_bw, "chr19"),
    axis_label=["Chemical Cleavage", "Prediction"],
    ax_scale = "norm"
)

save_fig_decorator("histplot_predictions_chr19")(corrplot)(
    load_from_bw(mnase_pred_bw, "chr19"),
    load_from_bw(chem_pred_bw, "chr19"),
    axis_label=["MNase-seq model", "Chemical Cleavage model",],
    ax_scale = "norm"
)

save_fig_decorator("histplot_labels_chr19")(corrplot)(
    load_from_bw(mnase_lab_bw, "chr19"),
    load_from_bw(chem_lab_bw, "chr19"),
    axis_label=["MNase-sequencing", "Chemical Cleavage"],
    ax_scale = "norm"
)


## Figure 1B — Genomic locus browser track with panel-to-panel correlations

In [ ]:
from matplotlib.lines import Line2D
def draw_bracket(fig, ax1, ax2, text=None, x=0.83, width=0.01, color="k"):
    p1 = ax1.get_position()
    p2 = ax2.get_position()

    y1 = p1.y0 + p1.height / 2
    y2 = p2.y0 + p2.height / 2

    fig.add_artist(Line2D([x, x], [y1, y2], transform=fig.transFigure, lw=1.5, c=color))
    fig.add_artist(Line2D([x-width, x], [y1, y1], transform=fig.transFigure, lw=1.5, c=color))
    fig.add_artist(Line2D([x-width, x], [y2, y2], transform=fig.transFigure, lw=1.5, c=color))

    if text is not None:
        fig.text(x + 0.005, (y1+y2)/2, text, va="center")

def pcorr(path1, path2):
    return np.corrcoef(
        load_from_bw(path1, chrom="chr19"),
        load_from_bw(path2, chrom="chr19"),
    )[0, 1]

def sig_extract_correl():
    pos, l = 15_980_000, 30_000
    x = np.arange(pos, pos+l)
    w_ratio = [10, 1]
    sigs_path = [mnase_lab_bw, mnase_pred_bw, chem_lab_bw, chem_pred_bw, mnase_mut_bw, chem_mut_bw, DNase_bw, wavelets_bw, atac1_bw]
    sigs_labels = ["MNase-seq", "MNase pred.", "Chemical cleavage", "Chemical pred.", "MNase ISM", "Chemical ISM", "DNase-seq", "Wavelets", "ATAC-seq"]
    colors = [
    "#D55E00", "#E69F00",  
    "#0072B2", "#56B4E9",  
    "#009E73", "#7FBF7B",  
    "#CC79A7",            
    "#332288",             
    "#AA4499"             
    ]

    fig, axs = plt.subplots(len(sigs_path), 2, figsize = (10, len(sigs_path)*0.4), sharex=True,
                            gridspec_kw={"width_ratios":w_ratio, "wspace":0}
                            )
    # fig.subplots_adjust(right=0.85)
    
    for i, s in enumerate(sigs_path):
        sig_tmp = load_from_bw(s, chrom="chr19", start=pos, end=pos+l)
        axs[i][0].plot(
            x,
            sig_tmp,
            color= colors[i]
        )
        axs[i][0].fill_between(x, sig_tmp, color=colors[i])

        axs[i][0].spines[["top", "right", "bottom"]].set_visible(False)
        axs[i][0].set_xlim(pos, pos+l)
        axs[i][1].axis("off")
        axs[i][0].set_ylabel(f"{sigs_labels[i]}", rotation=0, va="center", ha="right")
        axs[i][0].yaxis.set_label_coords(-0.08, 0.5)
        axs[i][0].set_yticks([np.max(sig_tmp)*0.75])

    for ax_idx in ((0, 1), (2, 3), (4, 5)):
        draw_bracket(
            fig,
            axs[ax_idx[0]][0],
            axs[ax_idx[1]][0],
            text=f"ρ = {pcorr(sigs_path[ax_idx[0]], sigs_path[ax_idx[1]]):.3f}",
            x=w_ratio[0]*0.085,
            color = "k"
        )

    axs[-1][0].spines["bottom"].set_visible(True)

    
    return fig, axs

save_fig_decorator("genome_browser_withCorrel")(sig_extract_correl)()

### Sanity check: predicted-track cross-correlation (MNase-seq model vs. chemical-cleavage model)

Used for the SUPP1C "predictions correlate with each other more than the raw assays" claim.

In [ ]:
np.corrcoef(
    load_from_bw(wavelet_pred_chem_bw, "chr1"),
    load_from_bw(wavelet_pred_mnase_bw, "chr1")
)

## SUPP Fig 1F — Autocorrelation of mutasome scores / nucleosome-repeat-length (NRL)

**Note**: the ≈187 bp NRL value quoted in the manuscript only appears here as a manually-chosen axis tick spacing (`range(0, 1001, 187)`), not as a printed/derived peak-lag value. Worth computing and printing the actual autocorrelation peak lag explicitly if you want this figure to *demonstrate* 187 bp rather than assume it.

In [ ]:
def plot_autocor(lag, cor, lim=1500, figax = None, **kwargs):
    if figax is None:
        fig , axs = plt.subplots(figsize=(1.5, 1.5))
    else:
        fig, axs = figax
    mask = (lag <= lim) & (lag>=0)
    axs.plot(lag[mask], cor[mask], **kwargs)
    return fig, axs

lag_mnase_mut, cor_mnase_mut = mf.autocorr(load_from_bw(mnase_mut_bw, "chr1"))
lag_chem_mut, cor_chem_mut = mf.autocorr(load_from_bw(chem_mut_bw, "chr1"))
lag_crossmut, cor_crossmut = mf.autocorr(load_from_bw(crossmut_bw, "chr1"))

In [ ]:
fig, axs = plot_autocor(
    lag_mnase_mut,
    cor_mnase_mut,
    lim=1500,
    color="#ff7f0e",
    label="MNase mutasome"
)
plot_autocor(
    lag_chem_mut,
    cor_chem_mut,
    lim=1500,
    figax=(fig, axs),
    color="#1f77b4",
    label="Chemical mutasome",
    alpha=0.7
)
plot_autocor(
    lag_crossmut,
    cor_crossmut,
    lim=1500,
    color="#2ca02c",
    figax=(fig, axs),
    label="Cross-mutasome score",
    alpha=0.7
)

axs.legend(bbox_to_anchor = (1.05, 1), loc="upper left")
plt.xlim(0, 50)
# fig.suptitle("Autocorrelation [chr1]")
axs.set_xticks(range(0, 51, 10))
plt.grid(True, linestyle="--", alpha=0.3)

plt.savefig(str(PATHS["fig_path"] / "mut_autocorr.svg"))
plt.show()

In [ ]:
lag_mnase_mut, cor_mnase_mut = mf.autocorr(load_from_bw(mnase_lab_bw, "chr1"))
lag_chem_mut, cor_chem_mut = mf.autocorr(load_from_bw(chem_lab_bw, "chr1"))

fig, axs = plot_autocor(
    lag_mnase_mut,
    cor_mnase_mut,
    lim=1500,
    color="#ff7f0e",
    label="MNase sequencing"
)
plot_autocor(
    lag_chem_mut,
    cor_chem_mut,
    lim=1500,
    figax=(fig, axs),
    color="#1f77b4",
    label="Chemical cleavage",
    alpha=0.7
)

axs.legend(bbox_to_anchor = [0.3, -0.65], loc="lower center")
plt.xlim(0, 1000)
# fig.suptitle("Occupancy Autocorrelation")
axs.set_xticks([0, 187, 561, 935])
plt.grid(True, linestyle="--", alpha=0.4)
plt.savefig(str(PATHS["fig_path"] / "nucleosomes_exp_autocorr.svg"))


In [ ]:
x = np.log10(mf.zscore(load_from_bw(mnase_mut_bw, "chr1"))+1)
y = np.log10(mf.zscore(load_from_bw(chem_mut_bw, "chr1"))+1)

fig = plt.figure(figsize=(2.2, 2))

gs = mpl.gridspec.GridSpec(4, 5, figure=fig, hspace=0.08, wspace=0.08,
                            width_ratios=[1, 1, 1, 1, 0.07])

ax_main  = fig.add_subplot(gs[1:4, 0:3])
ax_top   = fig.add_subplot(gs[0, 0:3], sharex=ax_main)
ax_right = fig.add_subplot(gs[1:4, 3], sharey=ax_main)
cax      = fig.add_subplot(gs[1:4, 4])

h = ax_main.hist2d(x, y, bins=50, cmap="YlOrRd", norm=mpl.colors.LogNorm())
h[3].set_rasterized(True)
ax_main.set_xlabel("Mnase ISM")
ax_main.set_ylabel("Chemical ISM")

cb = fig.colorbar(h[3], cax=cax)

ax_top.hist(x, bins=50, color="0.35")
ax_right.hist(y, bins=50, orientation="horizontal", color="0.35")
ax_top.set_yscale("log")
ax_right.set_xscale("log")

ax_top.tick_params(axis="x", labelbottom=False)
ax_right.tick_params(axis="y", labelleft=False)

for spine in ["top", "right", "left"]:
    ax_top.spines[spine].set_visible(False)
for spine in ["top", "right", "bottom"]:
    ax_right.spines[spine].set_visible(False)
ax_main.spines[["top", "right"]].set_visible(False)

plt.savefig(str(PATHS["fig_path"] / "muts_correlation.svg"), dpi=300)

## SUPP Fig 1G — Nucleosome islands vs. ISM hotspots (Venn diagram)

**Ambiguity flagged in the audit**: the original notebook contained two different Venn-diagram computations that could plausibly correspond to this panel — a genome-wide 3-way version (kept below, since it's the one that calls `savefig` and matches a genome-wide analysis) and a chr19-only 2-way version (moved to the Appendix). Neither one's visible output cleanly prints the exact 5919/26981/355698 counts from the published panel — worth re-running and checking against those numbers directly before trusting this is the right cell.

In [ ]:
import pandas as pd
import pyranges as pr
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from scipy.signal import find_peaks
import pyBigWig

chromosomes = [f"chr{i}" for i in range(1, 20)]

all_wave_mn_dfs = []
all_cmut_dfs = []

# --- 1. Iterate and detect peaks ---
for chrom in chromosomes:
    try:
        wave_mn_signal = load_from_bw(wavelet_pred_mnase_bw, chrom)
        cmut_signal = load_from_bw(crossmut_bw, chrom)
    except Exception as e:
        print("problem", e)
        continue

    wmn_peaks, _ = find_peaks(wave_mn_signal, prominence=4)
    cpeak_chrom, _ = find_peaks(cmut_signal, height=1, distance=20)

    if len(wmn_peaks) > 0:
        all_wave_mn_dfs.append(pd.DataFrame({
            "Chromosome": chrom, "Start": wmn_peaks, "End": wmn_peaks + 1, "Source": "Wave_MNase"
        }))

    if len(cpeak_chrom) > 0:
        all_cmut_dfs.append(pd.DataFrame({
            "Chromosome": chrom, "Start": cpeak_chrom, "End": cpeak_chrom + 1, "Source": "Crossmut"
        }))

# --- 2. Global concatenation ---
df_all = pd.concat(all_wave_mn_dfs + all_cmut_dfs, ignore_index=True)
pr_all = pr.PyRanges(df_all)

# --- 3. Spatial clustering (tolerance window) ---
window = 1000
pr_clustered = pr_all.cluster(slack=window)
df_clustered = pr_clustered.as_df()

# --- 4. Identification des combinaisons d'intersection par cluster ---
cluster_sources = df_clustered.groupby("Cluster")["Source"].apply(set)

# Input order for venn2: (10, 01, 11)
c_10 = sum(s == {"Wave_MNase"} for s in cluster_sources)
c_01 = sum(s == {"Crossmut"} for s in cluster_sources)
c_11 = sum(s == {"Wave_MNase", "Crossmut"} for s in cluster_sources)

# --- 5. 2-set Venn diagram ---
import numpy as np
from matplotlib_venn import venn2_unweighted

props = (c_10/sum_tmp, c_01/sum_tmp, c_11/sum_tmp)
f = lambda x: np.sqrt(x)
areas = tuple(map(f, props))  # aires adoucies pour la mise en page, indépendantes des valeurs affichées
plt.figure(figsize=(2, 2))
v = venn2_unweighted(
    subsets=props,
    subset_areas=areas,
    set_labels=('Wavelets', 'ISM peaks'),
    set_colors=('#332288', '#009E73'),
    subset_label_formatter=lambda x: f"{x:.0%}"
)

y_common = min(l.get_position()[1] for l in v.set_labels)
for l in v.set_labels:
    l.set_y(y_common)

plt.savefig(str(PATHS["fig_path"] / "waveletmnase_ism_span1000.svg"), dpi=300)


## Figure 1C — Multi-omic heatmaps at ISM hotspots, and Figure 1D — ISM/RNA-seq around TSS

**Flag from the audit**: the `n=388,694` NPRs quoted in the Fig. 1C caption is computed genome-wide with a combined MNase x chemical signal and `prominence=1` peak-calling in the Motif-discovery section further down (cell producing `final_nprs_df`). The heatmap built in this section instead calls `find_peaks` on the **MNase-only** mutasome track, restricted to `chr19`, with `prominence=5` — a different population from a different procedure. Worth checking whether the `n=388,694` figure caption number should instead reference this section's peak count, or whether this section should be switched to reuse `final_nprs_df`.

In [ ]:
rnam_bw = pbw.open(str(PATHS["rnaseq_minus_bw"]))
rnap_bw = pbw.open(str(PATHS["rnaseq_plus_bw"]))

rnam_bw_nbr = np.sum([np.nansum(rnam_bw.values(f"chr{chrom}", 0, -1, numpy=True)) for chrom in range(1, 20)])
rnap_bw_nbr = np.sum([np.nansum(rnap_bw.values(f"chr{chrom}", 0, -1, numpy=True)) for chrom in range(1, 20)])

coord_tss = pd.read_csv(PATHS["tss_coord_bed"], sep="\t", header=None, index_col=3)
coord_tss = coord_tss.iloc[:, :5]
coord_tss.columns = ["chrom", "p0", "p1", "name", "strand"]
coord_tss.drop("name", axis=1, inplace=True)

tss_class = pd.read_csv(PATHS["tss_classification"], sep="\t", index_col=0)
coord_tss = coord_tss.merge(tss_class, left_index=True, right_index=True)
cage_tss = pd.read_csv(PATHS["tss_annotation"], sep="\t", index_col=0)
coord_tss = coord_tss.merge(cage_tss, left_index=True, right_index=True)
coord_tss["TSSclassification"] = coord_tss["TSSclassification"].map({"yes": 1, "no": 0})


sparse_tss = mf.make_sparse_ones(coord_tss)
coord_tss = coord_tss.reset_index()


# coord_tss = coord_tss.merge(
#     pd.read_csv("/home/maxime/data/rnaseq/ENCFF827OZU.tsv", sep="\t"),
#      left_on="Transcript_name", right_on="transcript_id(s)", how="inner"
# )
# coord_tss

coord_tss = coord_tss[~coord_tss.isna().any(axis=1)]


def get_transcript(coord_tss, rnam_bw, nchrom):
    rnam_tmp = rnam_bw.values("chr1", 0, -1, numpy = True)
    rnam_tmp[np.isnan(rnam_tmp)] = 0
    coord_tmp = [np.arange(x[0], x[-1]) for x in mf.consecutive(np.where(rnam_tmp > 0)[0], 75)]
    return pd.DataFrame([(nchrom, x[0], x[-1]) for x in coord_tmp], columns=["chrom", "p0", "p1"])

mut_bw = pbw.open(str(PATHS["mnase_train_mutasome_bw"]))
size=1500
def get_strand(rnam_bw, mut_bw, strand, size=1500):
    rnam_tss_tmp = []
    mutm_tss_tmp = []
    for i in range(1, 20):
        coord_tss_tmp = coord_tss[(coord_tss.chrom == f"chr{i}") & (coord_tss.strand==strand)]
        if strand=="+":
            tss_ranges = mf.create_ranges(coord_tss_tmp.p0.values-size, coord_tss_tmp.p0.values+size).reshape(-1, 2*size)
        elif strand=="-":
            tss_ranges = mf.create_ranges(coord_tss_tmp.p1.values-size, coord_tss_tmp.p1.values+size).reshape(-1, 2*size)
        else:
            raise ValueError(f"Strand can't be {strand}")
        rnam_tmp = rnam_bw.values(f"chr{i}", 0, -1, numpy = True)
        rnam_tss_tmp.append(rnam_tmp[tss_ranges])
        mut_tmp = mf.zscore(mut_bw.values(f"chr{i}", 0, -1, numpy = True))
        mutm_tss_tmp.append(mut_tmp[tss_ranges])

    return rnam_tss_tmp, mutm_tss_tmp


strand_p = get_strand(rnap_bw, mut_bw, "+", size)
strand_m = get_strand(rnam_bw, mut_bw, "-", size)

mut_tss = np.vstack([
    np.vstack(strand_p[1]),
    np.vstack(strand_m[1])[:, ::-1]
]
)

rna_tss = np.vstack([
    np.nan_to_num(np.vstack(strand_p[0]), 0),
    np.nan_to_num(np.vstack(strand_m[0])[:, ::-1], 0)
]
)

idx = mut_tss.mean(axis=1).argsort()
rna_data = np.log10(rna_tss + 1e-5)


In [ ]:
def tss_heatmap():

    fig, (rna_plot, mut_plot) = plt.subplots(
        1, 2, figsize=(6, 4)
    )

    # --- IMAGE RNA ---
    rna_im = rna_plot.imshow(
        rna_data[idx],
        cmap="seismic",
        aspect="auto",
        norm=mpl.colors.Normalize(vmin=-4, vmax=-1)  
    )

    # --- IMAGE MUT ---
    mut_im = mut_plot.imshow(
        mut_tss[idx],
        cmap="seismic",
        aspect="auto",
        norm=mpl.colors.Normalize(vmin=-1, vmax=1) 
    )

    # --- Ligne du TSS si 'size' = index du TSS ---
    rna_plot.axvline(size, color="k", lw=1, ls="--")
    mut_plot.axvline(size, color="k", lw=1, ls="--")

    # --- Axes / titres ---
    rna_plot.set_title("RNA-seq around TSS", fontsize=10)
    mut_plot.set_title("ISM around TSS", fontsize=10)

    for ax in (rna_plot, mut_plot):
        ax.set_xlabel("Position relative to TSS (bp)", fontsize=10)
        ax.tick_params(axis="both", labelsize=8)
        # enlever les cadres sup & droite
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # Option : ticks en -1kb / TSS / +1kb si tu as 'size'
    L = rna_data.shape[1]
    xticks = [0, size, L - 1]
    xticklabels = [f"-{size} bp", "TSS", f"+{L-size} bp"]
    for ax in (rna_plot, mut_plot):
        ax.set_xticks(xticks)
        ax.set_xticklabels(xticklabels, rotation=0)

    # Y-axis : garder les labels seulement pour le premier
    mut_plot.set_yticks([])
    # rna_plot.set_ylabel("Genes", fontsize=10)

    # --- Colorbars fines et propres ---
    cbar_rna = fig.colorbar(
        rna_im, ax=rna_plot,
        fraction=0.046, pad=0.04
    )

    # positions des ticks (toujours en log10 car c’est ce que montre imshow)
    ticks = [-4, -3, -2, -1]
    cbar_rna.set_ticks(ticks)

    # labels en puissance de 10
    cbar_rna.set_ticklabels([r"$10^{-4}$", r"$10^{-3}$", r"$10^{-2}$", r"$10^{-1}$"])

    # cbar_rna.set_label("RNA-seq normalized coverage", fontsize=9)
    cbar_rna.ax.tick_params(labelsize=8)

    cbar_mut = fig.colorbar(
        mut_im, ax=mut_plot,
        fraction=0.046, pad=0.04
    )
    # cbar_mut.set_label("Mutation Z-score", fontsize=9)
    cbar_mut.ax.tick_params(labelsize=8)

    return fig, (rna_plot, mut_plot)

save_fig_decorator("TSS_ISM_heatmap")(tss_heatmap)()

rnam_bw.close()
rnap_bw.close()
mut_bw.close()

In [ ]:
chrom = "chr19"

mut = load_from_bw(crossmut_bw, chrom)
dnase = load_from_bw(DNase_bw, chrom)

wavelets = load_from_bw(wavelets_bw, chrom)
atac = np.mean(np.vstack([load_from_bw(atac1_bw, chrom), load_from_bw(atac2_bw, chrom)]), axis=0)

lab_chem = mf.zscore(load_from_bw(chem_lab_bw, chrom))
lab_mnase = mf.zscore(load_from_bw(mnase_lab_bw, chrom))


peaks, prop = find_peaks(mut, height=1, distance=20)
nprs_coord = peaks[:, None] + np.arange(-1000, 1000).reshape(1, -1)

idx = np.argsort(prop["peak_heights"])[::-1]


mats = [
    lab_mnase[nprs_coord][idx],
    lab_chem[nprs_coord][idx],
    dnase[nprs_coord][idx],
    wavelets[nprs_coord][idx],
    atac[nprs_coord][idx],
    
]

In [ ]:
def multiom_hm():
    fig, axs = plt.subplots(2, 5, figsize=(8, 4), gridspec_kw={"wspace": 0.2, "height_ratios":[1, 10]})

    titles = [
        "MNase\nISM",
        "Chemical\nISM",
        "DNase-seq",
        "Wavelets",
        "ATAC-seq",
    ]
    vmins = [0, 0, None, 0, None,]
    vmaxs = [1, 1, None, 5, None]

    for i, (mat, ax) in enumerate(zip(mats, axs[0])):
        norm = mpl.colors.LogNorm() if i in [2, 4] else None
        im = axs[1][i].imshow(mat, aspect="auto", cmap="Reds", vmin=vmins[i], vmax=vmaxs[i], norm=norm)
        axs[1][i].set_yticks([])
        # axs[1][i].set_xticks([0, 1500, 3000])
        # ax.set_xticklabels(["-1.5kb", "0", "+1.5kb"])
        axs[1][i].set_xlabel(titles[i], fontsize=6)
        plt.colorbar(im, cax=axs[0][i], orientation="horizontal")
    
    axs[1][0].set_yticks([len(mat)-len(mat)%500])
    return fig, axs

save_fig_decorator("multiom_hm")(multiom_hm)()

## Motif discovery — Table 1 (XSTREME cluster composition) & Figure 2C (top clusters by fraction of NPRs)

**Flag from the audit**: Table 1's `Num` column (motif-instance counts, built below from `xstreme.tsv`/Tomtom) and Figure 2C's cluster ranking (`percent_nprs`, built further down from an independent per-NPR FIMO best-hit assignment) measure two conceptually different things and are **not** straightforwardly cross-checkable against each other. Neither the Methods text nor this notebook currently explains that distinction — worth a one-line note in the manuscript if both are meant to stand side by side.

The original notebook also contained a second, differently-formatted LaTeX table generator for this cluster table (writing to the same `xtsreme_clusters.tex` output). Its columns ("Total Motifs"/"De Novo"/"Known") don't match the published Table 1 format ("Num"/"DeCounKC"/"Enriched Reference Motifs"/"Aligned De Novo Targets"), so it looks superseded by the version kept here — moved to the Appendix rather than deleted.

In [ ]:
xstreme_global_path = PATHS["xstreme_dir"]
xstreme_all = pd.read_csv(
    xstreme_global_path / "xstreme.tsv", sep="\t", comment="#"
)
xstreme_all

In [ ]:
import json
from pathlib import Path
import re
import pandas as pd


def extract_pure_xstreme_annotations(html_path):
    """Extracts the biological composition of XSTREME clusters by resolving

    the 'similar known motifs' only for de novo matrices (db < 0).
    """
    html_content = Path(html_path).read_text(encoding="utf-8")

    # Capture the native JSON block
    json_match = re.search(r"var\s+data\s*=\s*(\{.*?\});", html_content, re.DOTALL)
    if not json_match:
        raise ValueError(
            "Could not locate the global 'data' variable in the HTML file."
        )

    data = json.loads(json_match.group(1))
    motifs_list = data.get("motifs", [])
    motifs_dict = {m["id"]: m for m in motifs_list}

    clusters_data = data.get("clusters", [])
    if not clusters_data:
        fallback_clusters = {}
        for m in motifs_list:
            c_id = m.get("cluster")
            if c_id is not None:
                fallback_clusters.setdefault(c_id, []).append(m["id"])
        clusters_data = [
            {"id": k, "motifs": v} for k, v in sorted(fallback_clusters.items())
        ]

    cluster_records = []

    for cluster in clusters_data:
        cluster_id = cluster.get("id")
        motif_ids = cluster.get("motifs", [])

        if not motif_ids:
            continue

        de_novo_count = 0
        known_count = 0
        resolved_tfs = []

        for m_id in motif_ids:
            motif_meta = motifs_dict.get(m_id)
            if not motif_meta:
                continue

            # Rigorous discrimination of provenance
            db_index = motif_meta.get("db", 0)
            is_known = db_index >= 0

            if is_known:
                known_count += 1
                # For an already-known motif (e.g. JASPAR), use its canonical name
                tf_name = motif_meta.get("alt") or motif_meta.get("id")
                if tf_name:
                    tf_name = tf_name.split("(")[0].strip()
                    resolved_tfs.append(tf_name)
            else:
                de_novo_count += 1
                # For a de novo motif (MEME/STREME), extract the "similar known motif" (Tomtom)
                best_match = motif_meta.get("best_match", {})
                tf_name = None
                if isinstance(best_match, dict):
                    tf_name = best_match.get("name")

                # If no Tomtom alignment is found, mark the annotation as absent
                if tf_name:
                    tf_name = tf_name.split("(")[0].strip()
                    resolved_tfs.append(tf_name)

        # Aggregate and compute frequencies of the resolved transcription factors
        if resolved_tfs:
            counts = pd.Series(resolved_tfs).value_counts()
            composition_str = ", ".join(f"{tf}({cnt})" for tf, cnt in counts.items())
        else:
            composition_str = "No annotated match"

        cluster_records.append(
            {
                "Cluster": cluster_id,
                "Total Motifs": len(motif_ids),
                "De Novo (N)": de_novo_count,
                "Known (N)": known_count,
                "Cluster Composition": composition_str,
            }
        )

    return pd.DataFrame(cluster_records)


# -------------------------------------------------------------------------
# Generate and structure the LaTeX code
# -------------------------------------------------------------------------
xstreme_html_file = (
    PATHS["xstreme_dir"] / "xstreme.html"
)

summary_df = extract_pure_xstreme_annotations(xstreme_html_file)

latex_df = summary_df.copy()


def format_latex_cell_lines(cell_value):
    if (
        pd.isna(cell_value)
        or str(cell_value).strip() in ["None", "Unknown", "No annotated match"]
    ):
        return str(cell_value)
    elements = [el.strip() for el in str(cell_value).split(",")]
    if len(elements) <= 1:
        return str(cell_value)

    inner_lines = " \\\\ ".join(elements)
    return rf"\makecell[l]{{{inner_lines}}}"


latex_df["Cluster Composition"] = latex_df["Cluster Composition"].apply(
    format_latex_cell_lines
)

raw_tabular = latex_df.to_latex(index=False, column_format="crrrX", escape=False)

lines = raw_tabular.split("\n")
structured_lines = []
caption_text = (
    r"\caption{XSTREME cluster resolution. For each independent cluster, the absolute number of "
    r"discovered de novo matrices is quantified alongside the frequency of their resolved, "
    r"highly-similar known transcription factor motifs. \label{tab:motif_clusters}}"
)

for line in lines:
    if r"\begin{tabular}{crrrX}" in line or r"\begin{tabularx}" in line:
        structured_lines.append(r"\begin{tabularx}{\textwidth}{crrrX}")
        structured_lines.append(caption_text + r" \\")
        continue
    if r"\toprule" in line:
        structured_lines.append(line)
        continue
    if "Cluster & Total Motifs" in line:
        structured_lines.append(line)
        structured_lines.append(r"\midrule")
        structured_lines.append(r"\endhead")
        continue

    structured_lines.append(line)

    if (
        "\\" in line
        and "begin" not in line
        and "end" not in line
        and "midrule" not in line
        and "toprule" not in line
        and "bottomrule" not in line
    ):
        structured_lines.append(r"\midrule")

latex_code_fixed = "\n".join(structured_lines).replace(
    r"\end{tabular}", r"\end{tabularx}"
)

# with open("xtsreme_clusters.tex", "w") as f:
#     f.write(latex_code_fixed)

In [ ]:
import json
import re
from pathlib import Path

# Read the raw file
html_content = Path(
    PATHS["xstreme_dir"] / "xstreme.html"
).read_text(encoding="utf-8")

# Extract and deserialize the global 'data' variable
data = json.loads(
    re.search(r"var\s+data\s*=\s*(\{.*?\});", html_content, re.DOTALL).group(1)
)

In [ ]:
import json
import re
from pathlib import Path
import pandas as pd

# 1. Read the HTML file and load the native JSON
html_path = Path(
    PATHS["xstreme_dir"] / "xstreme.html"
)
html_content = html_path.read_text(encoding="utf-8")
data = json.loads(
    re.search(r"var\s+data\s*=\s*(\{.*?\});", html_content, re.DOTALL).group(1)
)

# 2. Initialize the per-cluster collection structure
cluster_map = {}

for m in data["motifs"]:
    cluster_id = m.get("cluster")
    if cluster_id is None:
        continue

    if cluster_id not in cluster_map:
        cluster_map[cluster_id] = {
            "jaspar_motifs": [],
            "tomtom_motifs": [],
            "de_novo_count": 0,
            "jaspar_count": 0,
        }

    is_jaspar = str(m.get("pgm", "")).strip().lower() == "sea"

    if is_jaspar:
        cluster_map[cluster_id]["jaspar_count"] += 1
        tf_name = m.get("alt") or m.get("id")
        if tf_name:
            cluster_map[cluster_id]["jaspar_motifs"].append(tf_name.strip())
    else:
        cluster_map[cluster_id]["de_novo_count"] += 1
        tomtom_list = m.get("tomtom_matches", [])

        if tomtom_list and isinstance(tomtom_list, list):
            for match in tomtom_list:
                tf_name = match.get("alt") or match.get("id")
                if tf_name:
                    cluster_map[cluster_id]["tomtom_motifs"].append(
                        tf_name.strip()
                    )
        else:
            consensus_seq = m.get("consensus")
            if consensus_seq:
                cluster_map[cluster_id]["tomtom_motifs"].append(
                    consensus_seq.strip()
                )


def generate_tomtom_frequency_list(motifs_list):
    if not motifs_list:
        return "-"

    counts = pd.Series(motifs_list).value_counts()
    counts = counts.sort_index().sort_values(ascending=False)

    formatted_elements = []
    for tf, cnt in counts.items():
        if cnt > 1:
            formatted_elements.append(f"{tf}({cnt})")
        else:
            formatted_elements.append(f"{tf}")

    return "; ".join(formatted_elements)


# 3. Build the DataFrame rows
cluster_records = []

for cluster_id, info in sorted(cluster_map.items()):
    jaspar_unique = sorted(list(set(info["jaspar_motifs"])))
    jaspar_str = "; ".join(jaspar_unique) if jaspar_unique else "-"

    total_de_novo = info["de_novo_count"]
    total_jaspar = info["jaspar_count"]
    total_motifs = total_de_novo + total_jaspar

    cluster_records.append(
        {
            "Cluster": cluster_id,
            "Total Motifs": total_motifs,
            "De Novo Count": total_de_novo,
            "Known Count": total_jaspar,
            "Enriched Reference Motifs (JASPAR)": jaspar_str,
            "Aligned De Novo Targets (Tomtom)": generate_tomtom_frequency_list(
                info["tomtom_motifs"]
            ),
        }
    )

summary_df = pd.DataFrame(cluster_records)

# 4. Ultra-compact geometry spec for the numeric columns
column_spec = "w{c}{0.5cm}w{c}{0.5cm}w{c}{0.7cm}w{c}{0.5cm}XX"

# Explicit use of the column format
raw_tabular = summary_df.to_latex(
    index=False, column_format=column_spec, escape=False
)

lines = raw_tabular.split("\n")
structured_lines = []
caption_text = (
    r"\caption{Comprehensive XSTREME cluster composition. \label{tab:motif_clusters}}"
)

for line in lines:
    if r"\begin{tabular}{" in line or r"\begin{tabularx}" in line:
        # Use .format() to cleanly isolate injection of the 'column_spec' variable
        begin_line = r"\begin{tabularx}{\textwidth}{" + column_spec + r"}"
        structured_lines.append(begin_line)
        structured_lines.append(caption_text + r" \\")
        continue
    if r"\toprule" in line:
        structured_lines.append(line)
        continue
    if "Cluster &" in line or "Total Motifs &" in line:
        # Headers condensed to a single letter to fit the 0.5cm space allocation
        # C = Cluster, T = Total, DN = De Novo, K = Known
        short_header = "C & T & DN & K & Enriched Reference Motifs (JASPAR) & Aligned De Novo Targets (Tomtom) \\\\"
        structured_lines.append(short_header)
        structured_lines.append(r"\midrule")
        structured_lines.append(r"\endhead")
        continue

    structured_lines.append(line)

    if (
        "\\" in line
        and "begin" not in line
        and "end" not in line
        and "midrule" not in line
        and "toprule" not in line
        and "bottomrule" not in line
    ):
        structured_lines.append(r"\midrule")

latex_code_fixed = "\n".join(structured_lines).replace(
    r"\end{tabular}", r"\end{tabularx}"
)

# Additional fix to clean up any leftover residue from pandas' original header
latex_code_fixed = re.sub(
    r"Total Motifs\s*&\s*De Novo Count\s*&\s*Known Count.*\\\\",
    "",
    latex_code_fixed,
)

# with open("xtsreme_clusters.tex", "w") as f:
#     f.write(latex_code_fixed)

In [ ]:
xstreme_all["de_novo"] = xstreme_all.SOURCE.isin(["STREME", "MEME"])
xstreme_all["SIM_MOTIF"] = xstreme_all.SIM_MOTIF.str.split(" ", expand=True).iloc[:, 0]
xstreme_all = xstreme_all.merge(
    jaspar_per_motif,
    how="left",
    left_on="SIM_MOTIF",
    right_on="motif_id"
)

rna = pd.read_csv(PATHS["rna_tpm_report"], sep="\t", header=1)
rna = rna[rna["Gene symbol"].notna()]
xstreme_all["tf_name_upper"] = xstreme_all["tf_name"].str.upper()
rna["Gene symbol_upper"] = rna["Gene symbol"].str.upper()
xstreme_all = xstreme_all.merge(
    rna[["Gene symbol_upper", "TPM"]],
    left_on="tf_name_upper", 
    right_on="Gene symbol_upper",
    how="left"
)
xstreme_all = xstreme_all.drop(columns=["tf_name_upper", "Gene symbol_upper"])
# xstreme_all = xstreme_all[((xstreme_all["TPM"]>=1) & (xstreme_all["de_novo"] == False)) | xstreme_all["de_novo"]]
xstreme_all["motif_id"] = xstreme_all["motif_id"].astype(str).str.replace(r"\s+", "", regex=True)
xstreme_all

In [ ]:
fasta_path = DEMO_DIR / "test" / "nprs.fasta"
fimo_root = PATHS["xstreme_dir"] / "fimo_by_motif"



In [ ]:
rows = []
pat = re.compile(r"(chr[0-9XYM]+):(\d+)-(\d+)")

for rec in SeqIO.parse(fasta_path, "fasta"):
    m = pat.search(rec.id)
    if not m:
        continue
    chrom, start, end = m.group(1), int(m.group(2)), int(m.group(3))
    rows.append({"seq_ID": rec.id, "Chromosome": chrom, "Start": start, "End": end})

npr_df = pd.DataFrame(rows)
gr_npr = pr.PyRanges(npr_df.rename(columns={"Start": "Start", "End": "End"}))


In [ ]:
fimo_files = sorted(glob.glob(os.path.join(fimo_root, "*", "fimo.tsv")))

best_chunks = [] 

best = npr_df[["seq_ID"]].copy()
best["motif_alt_id"] = pd.NA
best["p-value"] = 1.0               
best["norm_score"] = -np.inf       
best["score"] = np.nan
best["start"] = np.nan
best["stop"] = np.nan
best["strand"] = pd.NA
best["Chromosome"] = pd.NA

t0 = time.time()
for i, fp in enumerate(fimo_files):
    fimo = pd.read_csv(fp, comment="#", sep="\t")

    fimo["Chromosome"] = fimo["sequence_name"].astype(str).str.replace(r"^mm10_", "", regex=True)
    fimo["Start0"] = fimo["start"].astype(int) - 1
    fimo["End0"]   = fimo["stop"].astype(int)

    gr_fimo = pr.PyRanges(
        fimo.rename(columns={"Start0": "Start", "End0": "End"})[
            ["Chromosome", "Start", "End", "motif_alt_id", "score", "p-value", "start", "stop", "strand"]
        ]
    )
    hits = gr_fimo.join(gr_npr).df  

    if hits.empty:
        continue

    # norm_score = score / (stop - start)
    hits["norm_score"] = hits["score"] / (hits["stop"] - hits["start"] + 1)

    
    hits = hits.sort_values(["seq_ID", "p-value", "norm_score"], ascending=[True, True, False])
    cand = hits.drop_duplicates(["seq_ID"], keep="first")[
        ["seq_ID", "motif_alt_id", "p-value", "norm_score", "score", "start", "stop", "strand", "Chromosome"]
    ].copy()

    # Merge candidate
    tmp = best.merge(cand, on="seq_ID", how="left", suffixes=("_best", "_cand"))

    better = (
        (tmp["p-value_cand"].notna()) &
        (
            (tmp["p-value_cand"] < tmp["p-value_best"]) |
            ((tmp["p-value_cand"] == tmp["p-value_best"]) & (tmp["norm_score_cand"] > tmp["norm_score_best"]))
        )
    )

    # Update inplace
    best.loc[better.values, "motif_alt_id"] = tmp.loc[better, "motif_alt_id_cand"].values
    best.loc[better.values, "p-value"]      = tmp.loc[better, "p-value_cand"].values
    best.loc[better.values, "norm_score"]   = tmp.loc[better, "norm_score_cand"].values
    best.loc[better.values, "score"]        = tmp.loc[better, "score_cand"].values
    best.loc[better.values, "start"]        = tmp.loc[better, "start_cand"].values
    best.loc[better.values, "stop"]         = tmp.loc[better, "stop_cand"].values
    best.loc[better.values, "strand"]     = tmp.loc[better, "strand_cand"].values
    best.loc[better.values, "Chromosome"] = tmp.loc[better, "Chromosome_cand"].values

    mf.loadbar(i, len(fimo_files), t0)      


In [ ]:
best.to_csv(xstreme_global_path / "best_fimo_match.csv", index=False)

In [ ]:
save_best = pd.read_csv(PATHS["xstreme_dir"] / "best_fimo_match.csv")
cols = [x for x in save_best.columns if x not in ["ALT_ID", "EVALUE", "tf_name", "CLUSTER", "cluster", "TPM", "de_novo", "SOURCE", "ID"]]
save_best[cols]


In [ ]:
best = save_best.copy()

def make_merge_key(mid):
    mid = str(mid).strip()
    if "STREME" in mid:
        # "3-WAAAAAAAAAAAAAAASAA-STREME-3" → "STREME-3"
        m = re.search(r"(STREME-\d+)$", mid)
        return m.group(1) if m else pd.NA
    elif "MEME" in mid:
        m = re.search(r"(MEME-\d+)$", mid)
        return m.group(1) if m else pd.NA
    else:
        # "MA0753.3-ZNF740" → "ZNF740"
        return re.sub(r"^MA[\d\.]+\-", "", mid)

###############################################################################
# 3) Build merge_key on `best`, then join xstreme_all
###############################################################################

tok = best["motif_alt_id"].astype(str).str.split("-")

has_meme   = tok.apply(lambda x: "MEME"   in x if isinstance(x, list) else False)
has_streme = tok.apply(lambda x: "STREME" in x if isinstance(x, list) else False)

best["source_guess"] = np.select(
    [has_meme, has_streme],
    ["MEME",   "STREME"],
    default="JASPAR"
)

# de novo → keep last 2 tokens e.g. "MEME-2"; JASPAR → last token e.g. "MA0139.2"
best["merge_key"] = np.where(
    best["source_guess"].isin(["MEME", "STREME"]),
    best["motif_alt_id"].astype(str),          # STREME-3, MEME-2 → keep as-is
    best["motif_alt_id"].astype(str).str.replace(r"^MA[\d\.]+\-", "", regex=True)  # MA0538.2-daf-12 → daf-12
)
best["merge_key"] = best["motif_alt_id"].apply(make_merge_key)

In [ ]:
motifs = parse_meme_file(PATHS["xstreme_dir"] / "combined.meme")

df_motifs = pd.DataFrame([
    {
        "motif_id": m["motif_id"],
        "alt_name": m["alt_name"],
        "w": m["w"],
        "nsites": m["nsites"],
        "evalue": m["evalue"],
        "pwm": m["pwm"],
    }
    for m in motifs
])

In [ ]:
###############################################################################
# 4) Prepare xstreme_all join key (ALT_ID for de novo, ID for JASPAR)
###############################################################################

# xstreme_all["ID"] already looks like "MEME-2" / "STREME-23" / "MA0139.2"
# ALT_ID is the secondary key — use ID as the single right key
xstreme_cols = ["ID", "ALT_ID", "EVALUE", "tf_name", "CLUSTER", "cluster", "TPM", "de_novo", "SOURCE"]

best2 = best.merge(
    xstreme_all[xstreme_cols],
    how="left",
    left_on="merge_key",
    right_on="ALT_ID"
)

# Rename to avoid confusion with motif_id from df_motifs
best2 = best2.rename(columns={"ID": "xstreme_ID"})

In [ ]:
###############################################################################
# 5) Deduplicate — keep best p-value per seq_ID
###############################################################################

best2 = best2.sort_values("p-value", ascending=True).drop_duplicates("seq_ID")

assert best2["seq_ID"].is_unique, (
    f"Duplicates remaining: {best2['seq_ID'].duplicated().sum()}"
)

In [ ]:
assert best2["seq_ID"].is_unique, (
    f"best2 must be 1 row per seq_ID, duplicates: {best2['seq_ID'].duplicated().sum()}"
)

N_nprs = best2["seq_ID"].nunique()
df = best2.copy()
df["CLUSTER"] = df["CLUSTER"].astype(str)

# 1. Compute EVERYTHING for ALL clusters (no more is_other filter here!)
mk_counts = (
    df.groupby(["CLUSTER", "merge_key"], dropna=False)["seq_ID"]
      .nunique()
      .reset_index(name="n_nprs")
)

mk_best = (
    mk_counts.sort_values(["CLUSTER", "n_nprs"], ascending=[True, False])
             .drop_duplicates("CLUSTER", keep="first")
             .rename(columns={"merge_key": "merge_key_rep"})
)

mk_info = (
    df[["CLUSTER", "merge_key", "tf_name", "cluster"]]
      .drop_duplicates(["CLUSTER", "merge_key"])
      .rename(columns={"merge_key": "merge_key_rep"})
)

# Full archive table below, containing 100% of the clusters (can be saved or studied further)
full_plot_df = (
    df.groupby("CLUSTER", as_index=False)
      .agg(unique_site_count=("seq_ID", "nunique"))
      .merge(mk_best[["CLUSTER", "merge_key_rep"]], on="CLUSTER", how="left")
      .merge(mk_info, on=["CLUSTER", "merge_key_rep"], how="left")
      .drop_duplicates(subset=["CLUSTER"])
)

full_plot_df["percent_nprs"] = 100 * full_plot_df["unique_site_count"] / N_nprs
full_plot_df = full_plot_df.sort_values("percent_nprs", ascending=False).reset_index(drop=True)



topN = 16

plot_df = full_plot_df.head(topN).copy()
other_clusters_df = full_plot_df.iloc[topN:]

if not other_clusters_df.empty:

    n_other_sites = other_clusters_df["unique_site_count"].sum()
    

    n_other_motifs_total = df.loc[df["CLUSTER"].isin(other_clusters_df["CLUSTER"]), "merge_key"].nunique()

    other_row = pd.DataFrame([{
        "CLUSTER": "Other",
        "unique_site_count": int(n_other_sites),
        "percent_nprs": 100 * n_other_sites / N_nprs,
        "tf_name": f"Other ({n_other_motifs_total} motifs)",
        "cluster": "",
        "merge_key_rep": pd.NA,
    }])
    
    plot_df = pd.concat([plot_df, other_row], ignore_index=True)


N = len(plot_df)

fig, axs = plt.subplots(
    N, 2,
    sharex="col",
    sharey="col",
    gridspec_kw={"width_ratios": [3, 1]},
    figsize=(4, N * 0.3),     
    constrained_layout=True,  
)

max_x = float(plot_df["percent_nprs"].max())

for i, r in plot_df.reset_index(drop=True).iterrows():
    ax = axs[i, 1]

    is_other_row = (r["CLUSTER"] == "Other")
    bar_color = "lightgrey" if is_other_row else "dimgray"

    ax.barh(0, r["percent_nprs"], color=bar_color)

    if i != N - 1:
        ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
        ax.set_yticks([])
        ax.spines["bottom"].set_visible(False)
    else:
        ax.set_xticks(np.arange(0, max_x + 1, 20))

    ax.spines[["top", "right"]].set_visible(False)

    if is_other_row:
        ax.set_ylabel(str(r["tf_name"]), rotation=0, ha="left")
    else:
        cl = str(r["cluster"]).lstrip("cluster_") if pd.notna(r["cluster"]) else "?"
        tf = "Unknown" if pd.isna(r["tf_name"]) else str(r["tf_name"])
        ax.set_ylabel(f"Cl. {r.CLUSTER} ({tf})", rotation=0, ha="left") #Name

    ax.yaxis.set_label_position("right")


    axL = axs[i, 0]
    if is_other_row:
        axL.axis("off")
    else:
        motif_id = r["merge_key_rep"]


        m = df_motifs.loc[df_motifs["alt_name"] == motif_id, "pwm"]
        if len(m) == 0:
            axL.axis("off")
        else:
            pwm = m.values[0]
            entrop = mf.H(pwm)[:, None]
            pwm_df = pd.DataFrame((2 - entrop) * pwm, columns=["A", "C", "G", "T"])
            lm.Logo(pwm_df, ax=axL, color_scheme=jaspar_colors)

            if i != N - 1:
                axL.set_xticks([])
                axL.set_yticks([])
                axL.spines[["bottom", "top","right", "left"]].set_visible(False)
            else:
                axL.set_xticks(np.arange(0, 20, 5))
                axL.set_yticks([])

# plt.tight_layout()

plt.savefig(str(PATHS["fig_path"] / "XSTREME_synthesis.svg"))
plt.show()

## Figure 2D-I — ISM vs. PWM correlations (per-TF Spearman rho, incl. CTCF)

This panel's code lives in `src/fimo_by_motif_plots.py`, which is **not tracked in git** (see audit report) and still has its own hardcoded `/home/maxime/Nucleoseq/...` paths — not touched by this notebook cleanup. Fix/commit that file separately; it is not reproduced inline here to avoid duplicating a second, diverging copy of that logic.

# Figure 2 Boxplot

In [ ]:
single = pd.read_csv(PATHS["fimo_by_motif_single_csv"])
single = single.fillna(0)
xstreme_keyed = pd.read_csv(PATHS["xstreme_dir"] / "xstreme.tsv", sep="\t")
xstreme_keyed["merge_key"] = xstreme_keyed.ID + "-" + xstreme_keyed.ALT_ID

single_filtered = single.merge(xstreme_keyed[["merge_key", "CLUSTER", "EVALUE"]], left_on="motif_alt_id", right_on="merge_key")
single_filtered = single_filtered[single_filtered.norm_score > 1].copy()
single_filtered = single_filtered[single_filtered.groupby('CLUSTER')['CLUSTER'].transform('count') >= 500]
single_filtered.CLUSTER = single_filtered.CLUSTER.values.astype(int)

def boxplot_by_cluster(df, y_col, color, ax=None):
    if ax is None:
        ax = plt.gca()

    order = df.groupby("CLUSTER")[y_col].median().sort_values().index[::-1]
    data = [df.loc[df.CLUSTER == c, y_col].values for c in order]

    ax.boxplot(
        data,
        positions=range(len(order)),
        widths=0.6,
        whis=[5, 95],
        showfliers=False,
        patch_artist=True,
        boxprops=dict(facecolor=color, edgecolor="black", linewidth=0.5),
        medianprops=dict(color="black", linewidth=0.5),
        whiskerprops=dict(linewidth=0.5),
        capprops=dict(linewidth=0.5),
    )
    ax.set_xticks(range(len(order)), order, rotation=0)
    ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_xlim(-0.6, 10.6)
    ax.set_xlabel("Xstreme clusters")
    ax.set_ylabel("Correlation")
    return ax

plt.figure(figsize=(4, 2))
boxplot_by_cluster(single_filtered, "chem_rho", color="#1f77b4")
plt.savefig(str(PATHS["fig_path"] / "correlation_informationISM_chem.svg"))
plt.show()

plt.figure(figsize=(4, 2))
boxplot_by_cluster(single_filtered, "mnase_rho", color="#ff7f0e")
plt.savefig(str(PATHS["fig_path"] / "correlation_informationISM_mnase.svg"))
plt.show()

## Figure 2 SUPP — Average occupancy profiles per motif cluster (oriented on best match)

In [ ]:

top_clusters = plot_df.CLUSTER
conditions = []
for s in [mnase_lab_bw, mnase_pred_bw, chem_lab_bw, chem_pred_bw]:
    all_top = []
    for t in top_clusters:
        if t in ['nan', 'Other']:
            continue

        best_tmp = best2[best2["CLUSTER"] == int(float(t))]
        plus_rows  = best_tmp[best_tmp["strand"] == "+"]
        minus_rows = best_tmp[best_tmp["strand"] == "-"]

        plus = Parallel(n_jobs=-1)(
            delayed(load_from_bw)(s, r.Chromosome, int(r.start - 1000), int(r.start + 1000))
            for r in plus_rows.itertuples()
        )
        minus = Parallel(n_jobs=-1)(
            delayed(load_from_bw)(s, r.Chromosome, int(r.stop - 1000), int(r.stop + 1000))
            for r in minus_rows.itertuples()
        )

        stack = []
        if plus:
            stack.append(np.vstack(plus))
        if minus:
            stack.append(np.vstack(minus)[:, ::-1])  # reverse minus strand

        if stack:
            all_top.append(np.vstack(stack))

    conditions.append(all_top)

top_clusters = plot_df.CLUSTER.values[:-1]
alt_name_keys = plot_df.loc[plot_df["CLUSTER"].isin(top_clusters), "merge_key_rep"]
pwms = df_motifs.loc[df_motifs["alt_name"].isin(alt_name_keys), "pwm"]

In [ ]:
top_clusters = plot_df.CLUSTER
used_clusters = [t for t in top_clusters if t not in ['nan', 'Other']]
merge_key_by_cluster = plot_df.set_index("CLUSTER")["merge_key_rep"]
pwm_by_alt_name = df_motifs.set_index("alt_name")["pwm"]

pwms = [pwm_by_alt_name[merge_key_by_cluster[t]] for t in used_clusters]


def plot_meta(idx_inarray, color):
    fig, axs = plt.subplots(3, 5, sharex=True, sharey=False, figsize=(8, 4),
    gridspec_kw = {"wspace":0.5, "hspace":0.5})
    axs = axs.ravel()

    for i, a in enumerate(axs):
        if i >= len(conditions[idx_inarray]):
            a.axis("off")
            continue

        assay = np.mean(conditions[idx_inarray][i], axis=0)
        pred  = np.mean(conditions[idx_inarray + 1][i], axis=0)

        pwm = pwms[i]
        entrop = mf.H(pwm)[:, None]
        pwm_df = pd.DataFrame((2 - entrop) * pwm, columns=["A", "C", "G", "T"])

        axL = a.inset_axes([0, 1.05, 1, 0.3])
        lm.Logo(pwm_df, ax=axL, color_scheme=jaspar_colors)
        axL.axis("off")

        a.plot(assay, color="k", ls="--", alpha=0.7)
        a.plot(pred, color=color, alpha=0.7)
        a.spines[["top", "right"]].set_visible(False)
        a.set_yticks(np.round([np.min(pred), np.max(pred)], 2))
        a.set_xticks([0, 1000, 2000], [-1000, 0, 1000])

    return fig, axs


fig, axs = plot_meta(0, "darkorange")
fig.savefig(str(PATHS["fig_path"] / "metaplot_clusters_mnase.svg"))
fig.show()

fig, axs = plot_meta(2, "blue")
fig.savefig(str(PATHS["fig_path"] / "metaplot_clusters_chem.svg"))

fig.show()

## Figure 3A — Repeat-family enrichment within NPRs

Requires `nprs_df` (genome-wide NPR coordinates) from the Motif-discovery section above. **Re-enabled** the `savefig(...)` call, which was commented out in the original (see cleanup notes at the top).

In [ ]:
rm_pr = pr.PyRanges(rm.rename(
    columns={
        "genoName":"Chromosome",
        "genoStart" : "Start",
        "genoEnd" : "End"
    }))

nprs_df = pd.read_csv(PATHS["nprs_df"])
nprs_pr = pr.PyRanges(
    nprs_df
)


hits_with_uniqueseq = nprs_pr.join(rm_pr, how="left")
hits_df_wu = hits_with_uniqueseq.df


cols_to_fix = ["repClass", "repName", "repFamily"]
for col in cols_to_fix:
    hits_df_wu[col] = hits_df_wu[col].replace([-1, "-1"], pd.NA)
    hits_df_wu[col] = hits_df_wu[col].fillna("unique")


hits_df_wu["overlap_len"] = (
    hits_df_wu[["End", "End_b"]].min(axis=1) - 
    hits_df_wu[["Start", "Start_b"]].max(axis=1)
).clip(lower=0).fillna(0)


priority_map = {
    "SINE": 1, "LINE": 1, "LTR": 1, "DNA": 1, 
    "Satellite": 2, 
    "Simple_repeat": 3, "Low_complexity": 3,
    "Unknown": 4, "unique": 5
}
hits_df_wu["priority_score"] = hits_df_wu["repClass"].map(lambda x: priority_map.get(x, 4))


hits_df_wu["npr_id"] = (
    hits_df_wu["Chromosome"].astype(str) + "_" + 
    hits_df_wu["Start"].astype(str) + "_" + 
    hits_df_wu["End"].astype(str)
)


final_nprs_df = (
    hits_df_wu.sort_values(
        by=["overlap_len", "priority_score"], 
        ascending=[False, True]
    )
    .drop_duplicates(subset=["npr_id"])
)

print(f"Shape final : {final_nprs_df.shape}") 


In [ ]:
nprs_on_rm_df = final_nprs_df[["repName", "repClass", "repFamily"]].value_counts()
rm_counts = rm[["repName", "repClass", "repFamily"]].value_counts()



merged_df = pd.concat(
    [nprs_on_rm_df, rm_counts], 
    axis=1, 
    keys=["npr_overlaps", "genomic_total"]
)


merged_df = merged_df.reset_index()
merged_df = merged_df.fillna(0)

In [ ]:

df = merged_df.reset_index()
df["percentage"] = (df["npr_overlaps"] / df["genomic_total"]) * 100
df = df[df["percentage"] > 0].copy()


valid_classes = ["DNA", "LINE", "SINE", "Satellite", "Simple_repeat", "LTR", "Low_complexity"]
df["repClass"] = df["repClass"].apply(lambda x: x if x in valid_classes else "Other")


palette_colors = [
    "#000000",  
    "#E31A1C", 
    "#1F78B4",  
    "#33A02C",  
    "#FF7F00",  
    "#6A3D9A",  
    "#B15928",  
    "#E7298A",  
]


unique_classes = sorted(df["repClass"].unique())
color_map = dict(zip(unique_classes, palette_colors[:len(unique_classes)]))


df["color"] = df["repClass"].map(color_map)

def _union_len(starts, ends):
    total = 0
    cur_s, cur_e = starts[0], ends[0]
    for s, e in zip(starts[1:], ends[1:]):
        if s <= cur_e:
            if e > cur_e: cur_e = e
        else:
            total += cur_e - cur_s
            cur_s, cur_e = s, e
    total += cur_e - cur_s
    return total


def family_union_bp(rm):
    def per_chr(g):
        idx = np.argsort(g["genoStart"].values)
        return _union_len(g["genoStart"].values[idx], g["genoEnd"].values[idx])
    cov = rm.groupby(["repName","genoName"]).apply(per_chr).groupby("repName").sum()
    return cov  

cov_union = family_union_bp(rm)                          
f_cov = (cov_union / GENOME_LEN).rename("frac_cov")      
df = df.merge(f_cov, left_on="repName", right_index=True)

L_npr = int((nprs_df.End - nprs_df.Start).mean())
N = int(GENOME_LEN / L_npr)
n = len(nprs_df)



def pval_rep_npr(repname):
    row = df[df["repName"] == repname]
    if row.empty: return 0
    K = int(round(row["frac_cov"].iat[0] * N))
    x = int(row["npr_overlaps"].iat[0])
    
    return hypergeom.logsf(x - 1, N, K, n)

df["log_p"] = df["repName"].apply(pval_rep_npr)                 
df["minus_log10_pval"] = -df["log_p"]/np.log(10)


LOG_FLOOR = -1e6                    
logp = np.maximum(df["log_p"].to_numpy(), LOG_FLOOR)   

m = logp.size
order = np.argsort(logp)             
sorted_logp = logp[order]
ranks = np.arange(1, m+1)

# q_i(sorted) = min_{j>=i} p_j * m / j  --> en log : logq = logp + log(m/j)
sorted_logq = sorted_logp + np.log(m / ranks)
sorted_logq = np.minimum.accumulate(sorted_logq[::-1])[::-1] 

logq = np.empty_like(sorted_logq)
logq[order] = sorted_logq

df["minus_log10_q"] = -logq / np.log(10) 




In [ ]:
r

In [ ]:
def repeat_enrichment():
    y = "minus_log10_q"
    fig, ax = plt.subplots(figsize=(8, 6)) 


    for repclass, group in df.groupby("repClass"):

        current_color = color_map[repclass]
        
        ax.scatter(
            group[y],
            group["genomic_total"],
            c=current_color,
            s=10, 
            label=repclass,
            alpha=0.7
        )


    for r in df.itertuples():
        label_condition = (
            ((r.repClass == "LTR" and getattr(r, y) >= 1) or
            (r.repClass == "LINE" and getattr(r, y) >= 2) or
            (r.repClass == "Simple_repeat" and getattr(r, y) >= 1) or
            (r.repClass == "SINE" and getattr(r, y) >= 1) or
            (r.repClass == "Low_complexity" and getattr(r, y) >= 1) or
            (r.repClass == "DNA" and getattr(r, y) >= 1)) and
            getattr(r, "genomic_total") >2e3
        )
        
        if label_condition:
            ax.text(
                getattr(r, y), 
                r.genomic_total, 
                r.repName, 
                fontsize=10, 
                c=r.color,
                ha='left',
                va='bottom'
            )

    ax.set_ylabel("Number of Elements (log scale)")
    ax.set_xlabel("-log10(q-value) [BH FDR]")

    x_bh = -np.log10(0.01)  
    ax.axvline(x_bh, ls="--", c="gray", lw=1, label="FDR 1%")

    ax.legend(title="repClass", loc="center left", bbox_to_anchor=(1.02, 0.5))
    
    plt.yscale("log")
    plt.xscale("symlog")
    plt.xlim(left=-0.1)
    plt.tight_layout()
    
    return fig, ax


fig, ax = repeat_enrichment()
plt.savefig(str(PATHS["fig_path"] / "repeat_enrichment_full.svg"))
plt.show()



## Figure 3B — Top motif clusters per repeat class

In [ ]:
final_nprs_df["seq_ID"] = (
    final_nprs_df["Chromosome"].astype(str) + ":" +
    final_nprs_df["Start"].astype(str) + "-" +
    final_nprs_df["End"].astype(str)
)

motif_on_rm = pd.concat([
    best2[["seq_ID", "motif_alt_id", "CLUSTER", "tf_name", "SOURCE"]].set_index("seq_ID"),
    final_nprs_df[["seq_ID", "repName", "repClass", "repFamily"]].set_index("seq_ID")
], axis=1)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


df_min_1000 = motif_on_rm.groupby("repClass").filter(lambda x: len(x) >= 1000)

stats = (
    df_min_1000.groupby(["repClass", "CLUSTER"])
    .size()
    .reset_index(name="count")
    .sort_values(["repClass", "count"], ascending=[True, False])
)
top_10_per_class = stats.groupby("repClass").head(5)


class_order = (
    top_10_per_class.groupby("repClass")["count"]
    .sum()
    .sort_values(ascending=False)
    .index
)


sns.set_theme(style="white")
n_classes = len(class_order)
cols = 4
rows = (n_classes + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(8, 2 * rows),
                            gridspec_kw = {"hspace": 0.5}
)
axes = axes.flatten()

for i, rc in enumerate(class_order):
    ax = axes[i]
    data_class = top_10_per_class[top_10_per_class["repClass"] == rc]
    
    sns.barplot(
        data=data_class,
        x="CLUSTER",
        y="count",
        ax=ax,
        order=data_class["CLUSTER"],
        color="steelblue",
        edgecolor="black",
        linewidth=0.5
    )
    
    ax.set_title(rc, fontsize=10, pad=2)
    ax.set_xlabel("")
    ax.set_ylabel("")
    
    ax.tick_params(axis='x', labelsize=10, pad=-1)
    ax.tick_params(axis='y', labelsize=10)
    
    ax.grid(False)
    sns.despine(ax=ax)
    ax.set_yticks([data_class["count"].max()])

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# plt.tight_layout(h_pad=1.2, w_pad=1.0)
plt.savefig(str(PATHS["fig_path"] / "repeat_enrichment_cluster_hist.svg"))
plt.show()

## Figure 4A-B — Bound vs. unbound CTCF sites (chr19)

In [ ]:
import pandas as pd
import pyranges as pr

ctcf_chip_df = pd.read_csv(
    PATHS["ctcf_chip_bed"], 
    sep="\t"
)
ctcf_chip_df = ctcf_chip_df.iloc[:, [0, 1, 2, 4, 5]].copy()
ctcf_chip_df.columns = ["Chromosome", "Start", "End", "ChIP_Score", "Strand"]

# 2. Load and normalize the JASPAR sites (TSV)
ctcf_sites_df = pd.read_csv(
    PATHS["ctcf_jaspar_sites"], 
    sep="\t", 
    header=None
)
ctcf_sites_df = ctcf_sites_df.iloc[:, [0, 1, 2, 4, 5, 6]].copy()
ctcf_sites_df.columns = ["Chromosome", "Start", "End", "S1", "s2", "Strand"]



gr_chip = pr.PyRanges(ctcf_chip_df)["chr19"]
gr_sites = pr.PyRanges(ctcf_sites_df)["chr19"]


gr_joined = gr_sites.join(gr_chip, strandedness="same", suffix="_chip", how="left")
df_joined = gr_joined.as_df()



has_hit = df_joined["Start_chip"] != -1


overlap_start = df_joined.loc[has_hit, ["Start", "Start_chip"]].max(axis=1)
overlap_end = df_joined.loc[has_hit, ["End", "End_chip"]].min(axis=1)
overlap_length = overlap_end - overlap_start


is_valid = (~has_hit) | (has_hit & (overlap_length >= 12))

result_df = df_joined[is_valid].copy()
result_df = result_df.drop(columns=["Start_chip", "End_chip", "Strand_chip"])

In [ ]:
# check for duplicate JASPAR sites after the join
dupe_check = result_df.duplicated(subset=["Chromosome", "Start", "End", "Strand"], keep=False)
print("Total rows:", len(result_df))
print("Duplicate site rows:", dupe_check.sum())
print("Unique sites:", result_df.drop_duplicates(subset=["Chromosome","Start","End","Strand"]).shape[0])


In [ ]:
print("Total unique sites (chr19):", len(result_df))

# how many pass the S1 >= 850 JASPAR-quality threshold at all?
high_quality = result_df[result_df.S1 >= 850]
print("S1 >= 850:", len(high_quality))

# within that, how does ChIP_Score partition?
print("  ChIP_Score >= 10  :", (high_quality.ChIP_Score >= 10).sum())
print("  ChIP_Score < 0    :", (high_quality.ChIP_Score < 0).sum())
print("  0 <= ChIP_Score <10:", ((high_quality.ChIP_Score >= 0) & (high_quality.ChIP_Score < 10)).sum())

# and the full distribution, unfiltered by S1, for comparison
print("\nWithout S1 filter:")
print("  ChIP_Score >= 10  :", (result_df.ChIP_Score >= 10).sum())
print("  ChIP_Score < 0    :", (result_df.ChIP_Score < 0).sum())


In [ ]:
def get_ranges(result_df_tmp):

    s_p = result_df_tmp[(result_df_tmp["Strand"] == "+")]["Start"].values


    ranges_p = mf.create_ranges(s_p - 1000, s_p + 1000)
    ranges_p = ranges_p.reshape((-1, 2000))

    s_m = result_df_tmp[(result_df_tmp["Strand"] == "-")]["End"].values

    ranges_m = mf.create_ranges(s_m - 1000, s_m + 1000)
    ranges_m = ranges_m.reshape((-1, 2000))[:, ::-1]

    ranges = np.vstack([ranges_p, ranges_m])
    return ranges

result_df_tmp = result_df[(result_df["ChIP_Score"]>=10) & (result_df.S1>=850)].copy()
ranges_chip = get_ranges(result_df_tmp)

result_df_tmp = result_df[(result_df["ChIP_Score"]<0) & (result_df.S1>=850)].copy()
ranges_nochip = get_ranges(result_df_tmp)
print(ranges_nochip.shape, ranges_chip.shape)

In [ ]:
mnase_lab = load_from_bw(str(mnase_lab_bw), "chr19")
mnase_pred = load_from_bw(str(mnase_pred_bw), "chr19")
crossmut_vals = load_from_bw(str(crossmut_bw), "chr19")

In [ ]:
fig = plt.figure(figsize=(7.5, 3.5), dpi=300)

gs = fig.add_gridspec(1, 4, wspace=0.15, width_ratios=[1, 1, 1, 1])


ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1], sharey=ax0)
ax2 = fig.add_subplot(gs[0, 2], sharey=ax0)
ax3 = fig.add_subplot(gs[0, 3], sharey=ax0)
axs = [ax0, ax1, ax2, ax3]


pos1 = axs[1].get_position()
pos2 = axs[2].get_position()
pos3 = axs[3].get_position()


shift = 0.08 
axs[2].set_position([pos2.x0 + shift, pos2.y0, pos2.width, pos2.height])
axs[3].set_position([pos3.x0 + shift, pos3.y0, pos3.width, pos3.height])


line_color = "#7A2E22"
linewidth = 1.3

axs[0].plot(np.mean(mnase_lab[ranges_chip], axis=0), color=line_color, lw=linewidth)
axs[1].plot(np.mean(mnase_pred[ranges_chip], axis=0), color=line_color, lw=linewidth)
axs[2].plot(np.mean(mnase_lab[ranges_nochip], axis=0), color=line_color, lw=linewidth)
axs[3].plot(np.mean(mnase_pred[ranges_nochip], axis=0), color=line_color, lw=linewidth)


ax0.set_ylim(0.1, 0.8)
ax0.set_yticks([0.1, 0.8])
ax0.spines['left'].set_bounds(0.1, 0.8)

for i, ax in enumerate(axs):
    ax.set_xticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    
    if i > 0:
        ax.spines['left'].set_visible(False)
        ax.yaxis.set_visible(False)


for ax, label in zip(axs, ["Exp.", "Pred.", "Exp.", "Pred."]):
    ax.text(0.5, -0.15, label, transform=ax.transAxes, 
            ha='center', va='top', weight='bold', fontsize=11)

fig.text((axs[0].get_position().x0 + axs[1].get_position().x1)/2, 0.92, 
         f"Bound CTCF sites\nN={ranges_chip.shape[0]}", ha='center', va='bottom', fontsize=14, weight='normal')

fig.text((axs[2].get_position().x0 + axs[3].get_position().x1)/2, 0.92, 
         f"Unbound CTCF sites\nN={ranges_nochip.shape[0]}", ha='center', va='bottom', fontsize=14, weight='normal')

plt.savefig(str(PATHS["fig_path"] / "chip_ctcf_chr19_qual850.svg"))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


data_mut_ctcf = [
    np.mean(crossmut_vals[ranges_chip], axis=1), 
    np.mean(crossmut_vals[ranges_nochip], axis=1)
]
dic_stats = mf.choose_test_stat(*data_mut_ctcf)

stars = dic_stats['stars']
p_val = dic_stats['p_value']


fig, ax = plt.subplots(figsize=(4, 5), dpi=300)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.linewidth'] = 1.2


box_props = dict(linestyle='-', linewidth=1.5, color='#2C3E50')
median_props = dict(linestyle='-', linewidth=2.0, color='#E74C3C')
whisker_props = dict(linestyle='--', linewidth=1.2, color='#7F8C8D')

bp = ax.boxplot(
    data_mut_ctcf,
    showfliers=False,        
    whis=[5, 95],            
    widths=0.5
)


ax.set_xticklabels(['Bound', 'Unbound'], weight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', which='major', labelsize=16)


y_max_data = max([whisker.get_ydata()[1] for whisker in bp['whiskers']])
y_range = y_max_data - min([whisker.get_ydata()[0] for whisker in bp['whiskers']])


y_bar = y_max_data + (y_range * 0.05)
bar_height = y_range * 0.03
text_gap = y_range * 0.02


x1, x2 = 1, 2
ax.plot([x1, x1, x2, x2], [y_bar - bar_height, y_bar, y_bar, y_bar - bar_height], 
        color='black', lw=1.2)


annotation_text = f"{stars}\n(p = {p_val:.1e})" if p_val > 0 else f"{stars}\n(p < 1e-16)"
ax.text((x1 + x2) / 2, y_bar + text_gap, annotation_text, 
        ha='center', va='bottom', fontsize=10, weight='bold')


ax.set_ylim(ax.get_ylim()[0], y_bar + (y_range * 0.2))

plt.tight_layout()
# plt.savefig("figures_new/boxplot_ctcf_crossism.svg")
plt.show()

## Figure 4C — De novo CTCF motif (STREME) vs. JASPAR reference

In [ ]:
import io
import numpy as np


raw_data = """    0.340604	  0.065708	  0.427891	  0.165798	
  0.077825	  0.854547	  0.000000	  0.067628	
  0.010979	  0.983719	  0.000000	  0.005302	
  0.880311	  0.000000	  0.038957	  0.080732	
  0.032612	  0.352700	  0.587814	  0.026874	
  0.437949	  0.282425	  0.043632	  0.235994	
  0.916427	  0.004791	  0.026459	  0.052323	
  0.010713	  0.004605	  0.979345	  0.005337	
  0.602862	  0.007055	  0.367773	  0.022310	
  0.034460	  0.008999	  0.686653	  0.269888	
  0.044910	  0.001527	  0.945043	  0.008519	
  0.021453	  0.032517	  0.896279	  0.049751	
  0.119096	  0.733866	  0.028261	  0.118778	
  0.466438	  0.036806	  0.433823	  0.062933	
  0.080249	  0.403243	  0.203027	  0.313482	
  0.130051	  0.445131	  0.080686	  0.344132	
  0.489582	  0.177138	  0.231635	  0.101645	
  0.186201	  0.129986	  0.447654	  0.236160	
  0.498296	  0.128025	  0.194755	  0.178924	
  0.194696	  0.151747	  0.233172	  0.420384	"""



arr = np.loadtxt(io.StringIO(raw_data), dtype=np.float64)
def pwm_to_df(arr):
  entrop = mf.H(arr)[:, None]
  return pd.DataFrame((2-entrop)*arr, columns = mf.BASES[:4])

denovo_pwm =  pwm_to_df(arr)
motifs_japsar_tmp = parse_meme_file(PATHS["jaspar_meme"])
motifs_japsar = {}
for d in motifs_japsar_tmp:
    motifs_japsar[d["motif_id"]] = d

jaspar_ctcf = pwm_to_df(motifs_japsar["MA0139.2"]["pwm"])

In [ ]:
fig, axs = plt.subplots(2, 1, sharex=True, figsize=(3, 2), gridspec_kw={"hspace": 0.15})

lm.Logo(denovo_pwm, ax=axs[0], color_scheme=jaspar_colors)
axs[0].set_ylabel("De Novo")

lm.Logo(jaspar_ctcf, ax=axs[1], color_scheme=jaspar_colors)
axs[1].set_ylabel("JASPAR")

for ax in axs:
    ax.set_yticks([])
    ax.spines[["top", "right", "left"]].set_visible(False)

axs[1].set_xticks(np.arange(0, 20, 2))
# axs[1].tick_params(axis="x", length=10)
# axs[1].set_xlabel("Position")

plt.savefig(PATHS["fig_path"] / "CTCFcomparison.svg", bbox_inches="tight")
plt.show()

## Figure 4D — Repeat-class distribution of CTCF-like motif sites

**Flag from the audit**: this panel is captioned "functional CTCF binding sites" (implying the ChIP-verified, chr19-only bound set, n=1232, from 4A/B above), but the code below instead counts **genome-wide** STREME-1 motif matches (`sites_streme`) — a much larger (~130k), not-chr19-restricted, not-ChIP-filtered population. Worth confirming which population Fig. 4D is actually meant to show before publication.

In [ ]:
import pandas as pd
import pyranges as pr

# 1. Initial loading and filtering
sites_streme = pd.read_csv(
    PATHS["xstreme_dir"] / "streme_out" / "sites.tsv", 
    sep="\t"
)
sites_streme = sites_streme[sites_streme.motif_ALT_ID == "STREME-1"].copy()
sites_streme[["Chromosome", "Start", "End"]] = sites_streme.seq_ID.str.split(":", expand=True)


sites_streme["Start"] = sites_streme["Start"].astype(int)
sites_streme["End"] = sites_streme["End"].astype(int)


gr_streme = pr.PyRanges(sites_streme)
gr_repeats = rm_pr
gr_joined = gr_streme.join(gr_repeats, how="left", suffix="_repeat")
df_result = gr_joined.as_df()


df_result = df_result[["repClass"]].copy()
df_result["repClass"] = df_result["repClass"].astype(str)
allowed_classes = ["LINE", "SINE", "DNA", "LTR", "Low_complexity", "Simple_repeat"]
is_other = (~df_result["repClass"].isin(allowed_classes)) & (df_result["repClass"] != "-1")

df_result.loc[df_result["repClass"] == "-1", "repClass"] = "Unique"
df_result.loc[is_other, "repClass"] = "Other"


class_counts = df_result["repClass"].value_counts()

if "Unique" in class_counts:
    order = ["Unique"] + [c for c in class_counts.index if c != "Unique"]
    class_counts = class_counts.reindex(order)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import logomaker as lm


fig = plt.figure(figsize=(9, 2))
gs = gridspec.GridSpec(2, 2, width_ratios=[1.7, 1], hspace=0.4, wspace=0.3)


ax_logo1 = fig.add_subplot(gs[0, 0])  # Top left: de novo
ax_logo2 = fig.add_subplot(gs[1, 0], sharex=ax_logo1, sharey=ax_logo1)
ax_bar   = fig.add_subplot(gs[:, 1])  # Entire right column: histogram


logo_denovo = lm.Logo(denovo_pwm, ax=ax_logo1, color_scheme=jaspar_colors)
ax_logo1.set_ylabel("De novo", fontsize=10)

logo_jaspar = lm.Logo(jaspar_ctcf, ax=ax_logo2, color_scheme=jaspar_colors)
ax_logo2.set_ylabel("Reference", fontsize=10)


for ax in [ax_logo1, ax_logo2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xticks(np.arange(0, 21, 2))
    ax.tick_params(labelsize=10)



bars = ax_bar.bar(class_counts.index, class_counts.values, color="#34495E", width=0.6)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)
ax_bar.set_yticks([0, 25000, 50000], ["0", "25k", "50k"])
ax_bar.tick_params(axis='both', which='major', labelsize=10)


ax_bar.set_xticklabels(class_counts.index, rotation=45, ha='right')


for bar in bars:
    height = bar.get_height()
    ax_bar.annotate(f'{int(height)}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)

plt.savefig(str(PATHS["fig_path"] / "ctcf_distro_onRep.svg"), bbox_inches="tight")
plt.show()

## Figure 5D-E — Synthetic k-mer repeat titration (MSE vs. GC content)

In [ ]:
chem_kmers = np.load(PATHS["chem_run_dir"] / "synthetic_kmers_20260317_182039" / "synthetic_kmers.npz")
mnase_kmers = np.load(PATHS["mnase_run_dir"] / "synthetic_kmers_20260319_145329" / "synthetic_kmers.npz")
mnase_kmers.files



In [ ]:
def mse(pred, bg_pred, idx_rep = -1, axis=(1, 2)):
    return np.mean((pred[idx_rep]-bg_pred[None, :])**2, axis=axis)

In [ ]:
chem_mse = mse(bg_pred = chem_kmers['bg_predictions'], pred = chem_kmers['predictions'])
mnase_mse = mse(bg_pred = mnase_kmers['bg_predictions'], pred = mnase_kmers['predictions'])

In [ ]:
mnase_df = pd.read_csv(PATHS["mnase_run_dir"] / "synthetic_kmers_20260319_145329" / "kmers_plots" / "summary.csv")
chem_df = pd.read_csv(PATHS["chem_run_dir"] / "synthetic_kmers_20260317_182039" / "kmers_plots" / "summary.csv")

In [ ]:
import seaborn as sns
from adjustText import adjust_text
from collections import deque
import random

def get_reverse_complement(kmer):
    complement = {"A": "T", "T": "A", "C": "G", "G": "C", "N": "N"}
    return "".join(complement[base] for base in reversed(kmer))
    
def get_canonical_representative(kmer):
    k = len(kmer)
    variations = set()
    d = deque(kmer)
    for _ in range(k):
        current_rot = "".join(d)
        variations.add(current_rot)
        variations.add(get_reverse_complement(current_rot))
        d.rotate(1)
    return min(variations)


def swarmplot_personalized(df, nb_rep, title="K-mer analysis"):
    df_filtered = df.query(f"nb_rep == {nb_rep}").copy()
    
    df_agg = df_filtered.groupby("kmer").agg({
        'score': 'mean',
        'bg': 'first',
        'gc_content': 'mean' 
    }).reset_index()


    df_agg['canonical_rep'] = df_agg['kmer'].apply(get_canonical_representative)

    unique_groups = df_agg['canonical_rep'].unique()
    rng = np.random.default_rng(42)
    rng.shuffle(unique_groups)

    plt.figure(figsize=(4, 4))
    
    ax = sns.swarmplot(
        data=df_agg, 
        x='gc_content', 
        y='score', 
        hue='canonical_rep', 
        hue_order=unique_groups,
        palette='hls', 
        s=2,
        legend=False 
    )


    top_per_group = (df_agg
                     .sort_values('score', ascending=False)
                     .drop_duplicates('canonical_rep'))
    

    annot_df = top_per_group.head(30)

    sorted_gc_values = sorted(df_agg['gc_content'].unique())
    x_positions_dict = {gc: x for x, gc in enumerate(sorted_gc_values)}
    
    texts = []
    for _, row in annot_df.iterrows():
        x_graph = x_positions_dict[row['gc_content']]
        y_graph = row['score']
        

        t = ax.text(x_graph, y_graph, row['kmer'], 
                    fontsize=9, 
                    fontweight='bold')
        texts.append(t)


    adjust_text(texts, 
            expand_points=(1.5, 1.5),
            force_points=0.5)


    plt.title(f"{title} (rep={nb_rep})", fontsize=10)
    plt.xlabel("GC content", fontsize=10)
    plt.ylabel("MSE", fontsize=10)
    

    ax.yaxis.grid(True, linestyle='--', alpha=0.6)
    ax.xaxis.grid(True, linestyle='--', alpha=0.6)

    sns.despine()
        
    return df_agg


In [ ]:
mnase_agg = swarmplot_personalized(mnase_df.sort_values("gc_content"),nb_rep=10, title="MNase-seq")
plt.savefig(str(PATHS["fig_path"] / "synth_kmers_mnase.svg"))
plt.show()

chem_agg = swarmplot_personalized(chem_df.sort_values("gc_content"),nb_rep=10, title="Chemical")
plt.savefig(str(PATHS["fig_path"] / "synth_kmers_chem.svg"))
plt.show()

In [ ]:
chem_agg = swarmplot_personalized(chem_df.sort_values("gc_content"),nb_rep=10, title="Chemical cleavage")
plt.savefig(str(PATHS["fig_path"] / "synth_kmers_chem.svg"))
plt.show()

# G4

In [ ]:
g4 = pd.read_csv(DATA_DIR / "g4_intra_mm10.bed", sep="\t", header=None)

ncbi_to_ucsc_mapping = {
    "gi|371561115|gb|CM000994.2|": "chr1",
    "gi|371561114|gb|CM000995.2|": "chr2",
    "gi|371561113|gb|CM000996.2|": "chr3",
    "gi|371561112|gb|CM000997.2|": "chr4",
    "gi|371561111|gb|CM000998.2|": "chr5",
    "gi|371561110|gb|CM000999.2|": "chr6",
    "gi|371561109|gb|CM001000.2|": "chr7",
    "gi|371561108|gb|CM001001.2|": "chr8",
    "gi|371561107|gb|CM001002.2|": "chr9",
    "gi|371561106|gb|CM001003.2|": "chr10",
    "gi|371561105|gb|CM001004.2|": "chr11",
    "gi|371561104|gb|CM001005.2|": "chr12",
    "gi|371561103|gb|CM001006.2|": "chr13",
    "gi|371561102|gb|CM001007.2|": "chr14",
    "gi|371561101|gb|CM001008.2|": "chr15",
    "gi|371561100|gb|CM001009.2|": "chr16",
    "gi|371561099|gb|CM001010.2|": "chr17",
    "gi|371561098|gb|CM001011.2|": "chr18",
    "gi|371561097|gb|CM001012.2|": "chr19"
}

g4['chrom'] = g4[0].map(ncbi_to_ucsc_mapping)


In [ ]:
s_p = g4[(g4.chrom == "chr19") & (g4[5] == "+")][1].values
ranges_p = mf.create_ranges(s_p - 1000, s_p + 1000)
ranges_p = ranges_p.reshape((-1, 2000))

s_m = g4[(g4.chrom == "chr19") & (g4[5] == "-")][2].values
ranges_m = mf.create_ranges(s_m - 1000, s_m + 1000)
ranges_m = ranges_m.reshape((-1, 2000))[:, ::-1]

ranges = np.vstack([ranges_p, ranges_m])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def jointplot(sig, ranges, axs, hm_log=False, imshow_kw={}):
    """
    Renders a joint plot into specific subplots.
    axs must be an array of two Axes (one column: [ax_top, ax_bottom]).
    """
    sig = sig[ranges]
    
    axs[0].plot(np.mean(sig, axis=0))
    
    if hm_log: 
        sig = np.log10(sig + mf.eps)
        

    axs[1].imshow(sig, cmap="Reds", aspect="auto", **imshow_kw)

fig, axs_global = plt.subplots(2, 3, figsize=(6, 2), gridspec_kw={"hspace":0, "wspace":0.5})


jointplot(load_from_bw(crossmut_bw, "chr19"), ranges, axs=axs_global[:, 0], hm_log=True)
jointplot(load_from_bw(str(PATHS["mnase_lab_bw"]), "chr19"), ranges, axs=axs_global[:, 1], imshow_kw={"vmin": 0, "vmax": 1})
jointplot(load_from_bw(str(PATHS["chem_lab_bw"]), "chr19"),ranges, axs=axs_global[:, 2], imshow_kw={"vmin": 0, "vmax": 1})

axs_global[0][0].set_title("ISM")
axs_global[0][1].set_title("MNase-seq.")
axs_global[0][2].set_title("Chem.")
for a in axs_global[1][1:]:
    a.set_yticks([])
    a.set_xticks([])

for a in axs_global[0]:
    a.set_xticks([])

axs_global[1, 0].set_yticks([40_000], [f"{40000:.0e}"])
axs_global[1, 0].set_xticks([0, 1000, 2000], [-1000, 0, 1000])



plt.savefig(PATHS["fig_path"] / "g4.svg")
plt.show()

# Appendix — exploratory / superseded analyses

Cells below were present in the original notebook but either (a) don't correspond to a labeled figure/panel I could identify in the manuscript, (b) look superseded by a version already used in the main flow above, or (c) depend on a module that lives entirely outside this repository. Paths have been swapped to `PATHS[...]` where the target is knowable, but these are otherwise **unreviewed** — treat as reference, not working code.

### Alternate NPR peak-calling attempt (cross-mutasome geometric-mean peaks, chr1-19)

In [ ]:
chromosomes, peak, start, end, height = [], [], [], [], []


for chrom in [f"chr{i}" for i in range(1, 20)]:
    mnase_vals = prep_chromosome(mnase_mut, chrom)
    chem_vals = prep_chromosome(chem_mut, chrom)
    cross = np.sqrt(mnase_vals*chem_vals)
    peaks, props = find_peaks(cross, height=1, distance=20)

    for i, p in enumerate(peaks):
        chromosomes.append(chrom)
        peak.append(p)
        start.append(p-15)
        end.append(p+15)
        height.append(props["peak_heights"][i])

df = pd.DataFrame({
    "Chromosome":chromosomes,
    "Start":start,
    "End":end,
    "Height":height
})

# df.to_csv("/home/maxime/Nucleoseq/demo/test/nprs.csv", index=False)

### Cluster-size inspection of a `mut_wave_df.csv` (purpose unclear, ad hoc)

In [ ]:
import pandas as pd

df = pd.read_csv(PATHS["mut_wave_df_csv"])
# Boolean mask computed over the group
mask = df.groupby("Cluster")["Cluster"].transform("count") > 1

# Filter the original DataFrame
df_filtered = df[mask]
print(df_filtered)

### Alternate SUPP1G Venn attempt (chr19-only, 2-way: wavelet peaks vs. crossmut peaks)

Superseded in the main flow above by the genome-wide 3-way version (the one that calls `savefig`). Kept here since it's unclear which one the published panel actually used.

In [ ]:
# --- 6. Metrics and Venn diagram ---
n_wave = len(pr_wave)
n_cmut = len(pr_cmut)

intersection_wave = len(wave_matched_ids)
intersection_cmut = len(cmut_matched_ids)

only_wave = n_wave - intersection_wave
only_cmut = n_cmut - intersection_cmut
intersection = max(intersection_wave, intersection_cmut)

plt.figure(figsize=(4, 4))
venn2(
    subsets=(only_wave, only_cmut, intersection),
    set_labels=('', '')
)
plt.show()

In [ ]:
import pandas as pd
import pyranges as pr
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# --- 1. Build PyRanges objects with unique IDs ---
df_wave = pd.DataFrame({
    "Chromosome": ["chr19"] * len(wpeak), 
    "Start": wpeak, 
    "End": wpeak + 1,
    "ID_wave": range(len(wpeak))
})
df_cmut = pd.DataFrame({
    "Chromosome": ["chr19"] * len(cpeak), 
    "Start": cpeak, 
    "End": cpeak + 1,
    "ID_cmut": range(len(cpeak))
})

pr_wave = pr.PyRanges(df_wave)
pr_cmut = pr.PyRanges(df_cmut)

# --- 2. Intersection with tolerance via 'slack' ---
window = 1000

# pr.join with slack returns interval pairs that overlap or are within <= window distance
join_wave = pr_wave.join(pr_cmut, slack=window)
join_cmut = pr_cmut.join(pr_wave, slack=window)

# --- 3. Deduplicate based on unique identifiers ---
# Extract the unique IDs that found at least one partner
wave_matched_ids = set(join_wave.ID_wave) if not join_wave.empty else set()
cmut_matched_ids = set(join_cmut.ID_cmut) if not join_cmut.empty else set()

# --- 4. Extract the metrics for the diagram ---
n_wave = len(pr_wave)
n_cmut = len(pr_cmut)

intersection_wave = len(wave_matched_ids)
intersection_cmut = len(cmut_matched_ids)

only_wave = n_wave - intersection_wave
only_cmut = n_cmut - intersection_cmut
intersection = max(intersection_wave, intersection_cmut)

# --- 5. Plot ---
plt.figure(figsize=(6, 6))
venn2(
    subsets=(only_wave, only_cmut, intersection),
    set_labels=('Wave Peaks', 'Crossmut Peaks')
)

plt.title(f"PyRanges Overlap on chr19 (Slack: {window} bp)")
plt.show()

### Superseded XSTREME cluster LaTeX table generator

Writes a different table format ("Total Motifs"/"De Novo"/"Known" columns) to the same `xtsreme_clusters.tex` filename as the version kept in the main Motif-discovery section above, which matches the published Table 1 format instead. Run at most one of the two, since they'd overwrite each other's output otherwise.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def group_tf_names(group):
    """Condense multiple matches into a single clean string."""
    valid_tfs = group.dropna().replace("", np.nan).dropna().unique()
    if len(valid_tfs) == 0:
        return np.nan
    return "/".join(sorted(valid_tfs))


# -------------------------------------------------------------------------
# 1. Load the raw XSTREME files
# -------------------------------------------------------------------------
xstreme_global_path = Path(
    PATHS["xstreme_dir"]
)

# Read the main summary file (for clusters and consensus)
xstreme_all = pd.read_csv(
    xstreme_global_path / "xstreme.tsv", sep="\t", comment="#"
)
xstreme_all.columns = xstreme_all.columns.str.strip()
xstreme_all["ID"] = (
    xstreme_all["ID"].astype(str).str.replace(r"\s+", "", regex=True)
)
xstreme_all["de_novo"] = xstreme_all["SOURCE"].isin(["STREME", "MEME"])

# Lecture du fichier TOMTOM exhaustif (contient TOUS les matches significatifs)
tomtom_path = xstreme_global_path / "tomtom_out" / "tomtom.tsv"
if not tomtom_path.exists():
    # Alternative depending on XSTREME version (sometimes prefixed by the program name)
    tomtom_path = xstreme_global_path / "meme_tomtom_out" / "tomtom.tsv"

tomtom_df = pd.read_csv(tomtom_path, sep="\t", comment="#")
tomtom_df.columns = tomtom_df.columns.str.strip()

# Clean up identifiers in Tomtom (Query_ID = XSTREME ID, Target_ID = JASPAR ID)
tomtom_df["Query_ID"] = (
    tomtom_df["Query_ID"].astype(str).str.replace(r"\s+", "", regex=True)
)
tomtom_df["Target_ID"] = (
    tomtom_df["Target_ID"].astype(str).str.replace(r"\s+", "", regex=True)
)

# -------------------------------------------------------------------------
# 2. Resolve multiple matches (Tomtom <-> JASPAR)
# -------------------------------------------------------------------------
# Join Tomtom with jaspar_per_motif to retrieve the 'tf_name' of every target
tomtom_annotated = tomtom_df.merge(
    jaspar_per_motif, how="inner", left_on="Target_ID", right_on="motif_id"
)

# Aggregate by query motif (Query_ID) to concatenate all matched genes
tomtom_condensed = (
    tomtom_annotated.groupby("Query_ID")["tf_name"]
    .agg(group_tf_names)
    .reset_index()
)

# -------------------------------------------------------------------------
# 3. Inject the exhaustive annotations into the main table
# -------------------------------------------------------------------------
# Merge into the main table
xstreme_all = xstreme_all.merge(
    tomtom_condensed, how="left", left_on="ID", right_on="Query_ID"
)

# For known motifs imported directly from JASPAR that have no Tomtom Query_ID:
# If the motif is known, look up its name directly in the original JASPAR table
if "tf_name_y" in xstreme_all.columns:
    xstreme_all["tf_name"] = xstreme_all["tf_name_y"].fillna(
        xstreme_all["tf_name_x"]
    )
else:
    # Naming fallback when there's no column conflict
    xstreme_all["tf_name"] = xstreme_all["tf_name"].fillna(xstreme_all["ID"])

# Conditional replacement for the composition field
has_tf_match = xstreme_all["tf_name"].notna() & (
    xstreme_all["tf_name"].str.strip() != ""
)
xstreme_all["display_annotation"] = np.where(
    has_tf_match, xstreme_all["tf_name"], xstreme_all["CONSENSUS"]
)


# -------------------------------------------------------------------------
# 4. Per-cluster profiling & multi-page LaTeX code generation
# -------------------------------------------------------------------------
def profile_processed_clusters(df):
    df["EVALUE"] = pd.to_numeric(df["EVALUE"], errors="coerce")

    def get_top_annotations(x):
        counts = x.dropna().replace("", np.nan).dropna().value_counts()
        if counts.empty:
            return "Unknown"
        return ", ".join(
            f"{annot}({cnt})" for annot, cnt in counts.head(3).items()
        )

    summary = (
        df.groupby("CLUSTER")
        .agg(
            total_motifs=("display_annotation", "count"),
            de_novo_count=("de_novo", "sum"),
            known_count=("de_novo", lambda x: (~x).sum()),
            cluster_composition=("display_annotation", get_top_annotations),
            min_evalue=("EVALUE", "min"),
        )
        .reset_index()
    )
    return summary


summary_df = profile_processed_clusters(xstreme_all)

# Final formatting for the manuscript (ltablex)
latex_df = summary_df.iloc[:, :5].copy()
latex_df.columns = [
    "Cluster",
    "Total Motifs",
    "De Novo (N)",
    "Known (N)",
    "Cluster Composition",
]


def format_latex_cell_lines(cell_value):
    if pd.isna(cell_value) or cell_value in ["None", "Unknown"]:
        return cell_value
    elements = [el.strip() for el in str(cell_value).split(",")]
    if len(elements) <= 1:
        return cell_value
    return rf"\makecell[l]{{' \\\\ '.join(elements)}}"


latex_df["Cluster Composition"] = latex_df["Cluster Composition"].apply(
    format_latex_cell_lines
)

raw_tabular = latex_df.to_latex(index=False, column_format="crrrX", escape=False)

# Regenerate the long-table structure with separator rules
lines = raw_tabular.split("\n")
structured_lines = []
caption_text = (
    r"\caption{Exhaustive XSTREME cluster composition integrated with complete Tomtom alignments. "
    r"All degenerate TF matches per discovered motif are listed. \label{tab:motif_clusters}}"
)

for line in lines:
    if r"\begin{tabular}{crrrX}" in line or r"\begin{tabularx}" in line:
        structured_lines.append(r"\begin{tabularx}{\textwidth}{crrrX}")
        structured_lines.append(caption_text + r" \\")
        continue
    if r"\toprule" in line:
        structured_lines.append(line)
        continue
    if "Cluster & Total Motifs" in line:
        structured_lines.append(line)
        structured_lines.append(r"\midrule")
        structured_lines.append(r"\endhead")
        continue
    structured_lines.append(line)
    if (
        "\\" in line
        and "begin" not in line
        and "end" not in line
        and "midrule" not in line
        and "toprule" not in line
        and "bottomrule" not in line
    ):
        structured_lines.append(r"\midrule")

latex_code_fixed = "\n".join(structured_lines).replace(
    r"\end{tabular}", r"\end{tabularx}"
)

with open("xtsreme_clusters.tex", "w") as f:
    f.write(latex_code_fixed)

### Mappability / multi-mapped-vs-uniquely-mapped cosine-similarity exploration

Supports the SUPP1 title's "generalization within low-mappability regions" claim narratively, but doesn't obviously correspond to one of the labeled SUPP1 A-G panels. **Cannot run even after path fixes**: imports a `preprocess.default` function from a module (`/home/maxime/data/utils`) that lives entirely outside this repo and isn't part of any tracked or gitignored-but-present location — vendor or relocate that module first.

In [ ]:
from sys import path
path.insert(1, PATHS["external_utils_dir"])
from preprocess import default

with pbw.open(PATHS["mnase_multimapped_bw"]) as map_bw:
    multi_vals = default(map_bw.values("chr1", 0, -1, numpy = True))

with pbw.open(str(mnase_lab_bw)) as map_bw:
    uni_vals = map_bw.values("chr1", 0, -1, numpy = True)

with pbw.open(str(mnase_pred_bw)) as map_bw:
    pred_vals = map_bw.values("chr1", 0, -1, numpy = True)


In [ ]:

chrom="chr19"

# 1. Fetch data for the specific chromosome
with pbw.open(PATHS["mnase_multimapped_bw"]) as map_bw:
    # Note: Check if your BigWig uses 'chr1' or '1' as keys
    multi_vals = default(map_bw.values(chrom, 0, -1, numpy=True))

with pbw.open(str(mnase_lab_bw)) as map_bw:
    uni_vals = map_bw.values(chrom, 0, -1, numpy=True)

with pbw.open(str(mnase_pred_bw)) as map_bw:
    pred_vals = map_bw.values(chrom, 0, -1, numpy=True)

with pbw.open(PATHS["mappability_bw"]) as map_bw:
    map_vals = map_bw.values(chrom, 0, -1, numpy = True)

non_map = mf.consecutive(np.where(map_vals < 1)[0], 100)

def get_cos_sim(a, b):
    norm_prod = np.linalg.norm(a) * np.linalg.norm(b)
    return np.dot(a, b) / norm_prod if norm_prod > 0 else 0

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Define the region and the expanded coordinates
x_idx = non_map[5779]
# Creating a continuous range including buffers
x = np.arange(x_idx.min() - 500, x_idx.max() + 501)

# 2. Extract the specific signal slices
# These are the vectors actually used for plotting and math
y_multi = multi_vals[x]
y_pred = pred_vals[x]
y_uni = uni_vals[x]

# 3. Helper function for Cosine Similarity (Rigorous Calculation)
def get_cos_sim(a, b):
    norm_product = np.linalg.norm(a) * np.linalg.norm(b)
    if norm_product == 0: return 0.0
    return np.dot(a, b) / norm_product

# Calculate scores specifically for this window
current_multi_sim = get_cos_sim(y_multi[500:-500], y_pred[500:-500])
current_uni_sim = get_cos_sim(y_uni[500:-500], y_pred[500:-500])

# 4. Setup the figure with zero vertical space
fig, axs = plt.subplots(3, 1, sharex=True, figsize=(12, 6), gridspec_kw={'hspace': 0})

# 5. Plotting
axs[0].fill_between(x, y_multi, color='royalblue', alpha=0.8)
axs[1].fill_between(x, y_pred, color='crimson', alpha=0.8)
axs[2].fill_between(x, y_uni, color='forestgreen', alpha=0.8)

# 6. Adding text labels with the RECALCULATED scores
# Using transform=ax.transAxes ensures text stays in the same spot regardless of X coordinates
axs[0].text(0.01, 0.85, f"Multi-mapped (Cos Sim: {current_multi_sim:.4f})", 
            transform=axs[0].transAxes, fontweight='bold', fontsize=10)
axs[1].text(0.01, 0.85, "Prediction", 
            transform=axs[1].transAxes, fontweight='bold', color='crimson', fontsize=10)
axs[2].text(0.01, 0.85, f"Uni-mapped (Cos Sim: {current_uni_sim:.4f})", 
            transform=axs[2].transAxes, fontweight='bold', fontsize=10)

# 7. Formatting and Alignment checks
y_max = max(y_pred.max(), y_multi.max(), y_uni.max()) * 1.1

for i, ax in enumerate(axs):
    ax.grid(True, alpha=0.2, linestyle='--')
    ax.set_ylim(0, y_max)
    # Highlight the central region of interest (the 'non-map' part)
    ax.axvspan(x_idx.min(), x_idx.max(), color='gray', alpha=0.1, zorder=-1)
    
    if i < 2:
        ax.spines['bottom'].set_visible(False)
        ax.tick_params(bottom=False)

plt.xlabel("Genomic Position")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sys import path
path.insert(1, PATHS["external_utils_dir"])
from preprocess import default

# Initialize containers to store results from all chromosomes
all_multi_sims = []
all_uni_sims = []

# Loop through chromosomes 1 to 19
chromosomes = [f"chr{i}" for i in range(1, 20)]

# Helper function for Cosine Similarity
def get_cos_sim(a, b):
    norm_product = np.linalg.norm(a) * np.linalg.norm(b)
    if norm_product == 0:
        return 0.0
    return np.dot(a, b) / norm_product


for chrom in chromosomes:
    print(f"Processing {chrom}...")
    
    # 1. Fetch data for the specific chromosome
    with pbw.open(PATHS["mnase_multimapped_bw"]) as map_bw:
        # Note: Check if your BigWig uses 'chr1' or '1' as keys
        multi_vals = default(map_bw.values(chrom, 0, -1, numpy=True))

    with pbw.open(str(mnase_lab_bw)) as map_bw:
        uni_vals = map_bw.values(chrom, 0, -1, numpy=True)

    with pbw.open(str(mnase_pred_bw)) as map_bw:
        pred_vals = map_bw.values(chrom, 0, -1, numpy=True)

    with pbw.open(PATHS["mappability_bw"]) as map_bw:
        map_vals = map_bw.values(chrom, 0, -1, numpy = True)

    non_map = mf.consecutive(np.where(map_vals < 1)[0], 1)
    
    # Filter by size (> 1000)
    non_map_filtered = [x for x in non_map if len(x) > 500]

    # 3. Calculate similarities for this chromosome
    for x in non_map_filtered:
        target = pred_vals[x]
        all_multi_sims.append(get_cos_sim(multi_vals[x], target))
        all_uni_sims.append(get_cos_sim(uni_vals[x], target))

# 4. Final conversion and exclusion logic
multi_arr = np.nan_to_num(np.array(all_multi_sims))
uni_arr = np.nan_to_num(np.array(all_uni_sims))



In [ ]:
exclusion = ((multi_arr == 0) & (uni_arr == 0)) | (np.isclose(multi_arr, uni_arr, atol=0.001))


In [ ]:
from matplotlib.colors import LogNorm
multi_final = multi_arr[~exclusion]
uni_final = uni_arr[~exclusion]
plt.hist2d(uni_final, multi_final, cmap="Reds", bins=50, norm=LogNorm())
plt.colorbar()
plt.ylabel("Prediction/Multimapped consine similarity")
plt.xlabel("Prediction/Uniquely mapped consine similarity")
plt.title(f"Correlation on low mappability sequence (500+ bp) N={len(uni_final)}")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Set the aesthetic style
sns.set_theme(style="white")

# 1. Create the joint plot
# kind='hex' provides the 2D histogram feel
# bins='log' ensures we keep the logarithmic intensity scale for the density
g = sns.jointplot(
    x=uni_final, 
    y=multi_final, 
    kind='hist', 
    cmap="Reds", 
    # gridsize=40,# joint_kws={'bins': 'log'},
    marginal_kws={'color': 'firebrick', 'fill': True}
)

# 2. Add the Identity Line (y = x)
# This is crucial for your analysis: 
# Points above the line mean the Multimapped signal is more similar to the prediction.
# Points below the line mean the Uniquely mapped signal is better.
g.ax_joint.plot([0, 1], [0, 1], color='black', linestyle='--', linewidth=1.5, alpha=0.6, label="Identity Line ($y=x$)")

# 3. Refine labels and limits
g.set_axis_labels(
    "Cosine Similarity: Prediction vs Uniquely Mapped", 
    "Cosine Similarity: Prediction vs Multimapped", 
    fontsize=11, fontweight='bold'
)

# Since Cosine Similarity ranges from 0 to 1, we fix the axes
g.ax_joint.set_xlim(0, 1)
g.ax_joint.set_ylim(0, 1)

# 4. Add a descriptive title
# We use g.fig.suptitle because standard plt.title() overlaps with the marginal plot
g.fig.suptitle(
    f"Model Performance Comparison in Low Mappability Regions\nSequence length $> 500$ bp ($N={len(uni_final)}$)", 
    y=1.03, fontsize=13
)

# Add a colorbar manually (Seaborn hex jointplots don't include it by default)
cbar_ax = g.fig.add_axes([1.0, 0.1, 0.02, 0.7]) # [left, bottom, width, height]
plt.colorbar(plt.cm.ScalarMappable(norm=LogNorm(vmin=1, vmax=100), cmap="Reds"), cax=cbar_ax)
cbar_ax.set_ylabel('Density (Log Scale)', rotation=270, labelpad=15)

plt.tight_layout()
plt.show()
# plt.savefig("similarity_comparison_jointplot.png", dpi=300, bbox_inches='tight')